# Ocean emulator: UNet + CRPS ensemble

A monthly emulator of ACCESS-OM2 (1°, JRA55-do flux-forced) column **heat content (OHC)**,
**freshwater content**, **sea surface temperature (SST)**, **sea surface salinity (SSS)** and the **surface current**
(u, v). SST and the surface current are predicted so that the **coupler** can compute the surface fluxes (heat,
freshwater, wind stress) from the emulator's own surface, the way ACCESS-OM2's coupler does, instead of taking them
from ACCESS-OM2. That gives the emulated ocean the air-sea feedback:
if its SST drifts warm, it loses more heat to the air and cools back down. Given the last two months and this month's surface forcing, it predicts
the next month; chaining that step forward gives a free-running ocean.

This is the **ensemble** version. The deterministic version, with the same structure, is
`Ocean_Emulator_UNet.ipynb`.

| Ingredient | What it does |
|---|---|
| **UNet** | the network: maps (last 2 states, forcing, noise) to the change in the state |
| **Noise inputs** | 4 channels of fresh random noise every month, so every call gives a different, plausible next month |
| **CRPS loss** | scores a 4-member ensemble against the one true outcome: pulls members towards the truth *and* keeps them apart by as much as the outcome is really uncertain |
| **Spectral loss (× 2)** | each member's energy in each band of zonal wavenumber must match ACCESS-OM2's, so members stay sharp rather than smooth |
| **Hard global budget constraint** | after every step, the ocean-total heat and freshwater change is set exactly to what the surface fluxes put in |
| **Coupler** | ACCESS-OM2's own bulk formulae (ported from CICE), applied to the 3-hourly JRA55-do atmosphere, the sea ice (from ACCESS-OM2) and the emulator's own SST and surface current: no network |

**Training is in two stages:**
1. **Stage 1, deterministic warm-up** (no noise, MSE + 2 × spectral), rollouts lengthening 1 → 4 → 12 months.
2. **Stage 2, coupled CRPS ensemble**: start from stage 1, add the noise inputs (their weights start at zero, so
   the model starts out deterministic), and train with every member's fluxes from the coupler and its own surface,
   with CRPS + 2 × spectral: first 12-month rollouts, then months 25–36, then 49–60, of free runs.

Both stages train **coupled**, and both tests (the 2005–2018 skill test and the 20-year control) run coupled:
the emulator never sees a flux its own surface did not help make.

Why: the coupled prototypes (`runs/prototype_coupled_seed42`, `_seed43`) only drifted slowly in the 50-year
control (largest |z| 38–212 by year 50) and kept SST skill for the whole 14-year test. They used a statistical
(neural-network) coupler, replaced here by ACCESS-OM2's own bulk formulae.

**Reading the code:** nothing is imported from `src/`. Cells headed `IDENTICAL CODE EXISTS IN src/...`
are verbatim copies of the tested code in `src/`, placed where they are first used; every other cell is
written for this notebook only.

## 1. Setup

In [ ]:
# Standard Python
import os
from pathlib import Path

# Let the CUDA allocator grow memory segments instead of fragmenting them.
# This only works if it is set before anything touches the GPU, hence here at the top.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# Scientific Python
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

# PyTorch (the network, the losses) and Lightning (the training loop)
import lightning as L
import torch
import torch.nn as nn
import torch.nn.functional as F
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from torch.utils.checkpoint import checkpoint
from torch.utils.data import DataLoader

# Nothing is imported from src/: the cells headed "IDENTICAL CODE EXISTS IN src/..." are copies of it.
REPO = Path("/home/156/txs156/uom/OM2-emulator")

# Train on the GPU if there is one.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | device {DEVICE}")

## 2. Settings

Every setting is here.

In [ ]:
# ===================================== Data =========================================
DATA_PATH = "/g/data/nm47/txs156/OM2-emulator/data/1deg_ocean_heat_emulator_data_v5.nc"
# The ACCESS-OM2 1-degree grid: cell-centre latitude and longitude at every cell, land included (the
# data file masks them over land). Used for the length of every grid row (the spectral loss and spectra).
GRID_PATH = "/g/data/ik11/grids/ocean_grid_10.nc"

# State variables: predicted by the model, then fed back in as its input for the next month.
# SST and SSS (MOM ocean_month, same run; added in the v4 file) and the surface current u_surf, v_surf (top level,
# averaged onto the t-cells; added in the v5 file): the coupler computes the surface fluxes from the SST and the
# wind relative to the current, as ACCESS-OM2's does.
PROGNOSTIC = ["ocean_heat_content_2d", "ocean_freshwater_content_2d", "sst", "sss", "u_surf", "v_surf"]

# Forcing variables: the surface fluxes, never predicted. Both stages and both tests take them from the coupler
# (section 5).
FORCING = ["total_surface_heat_flx", "ocean_freshwater_flux", "tau_x", "tau_y"]

# ===================================== Dates ========================================
# All dates are "YYYY-MM"; all ranges include both ends.
TIME_AXIS = ("1970-01", "2018-12")      # months loaded from the file
NORM_FIT = ("1970-01", "2000-12")       # months the climatology (mean, std) is computed from
TRAIN = ("1970-02", "2000-12")          # training samples start here; their whole window must end by 2000-12
VALID = ("2001-01", "2001-12")          # validation samples start in these months
TEST_START = "2005-02"                  # skill test: start from the true state of this month ...
TEST_END = "2018-11"                    # ... and predict up to one month after this (2018-12)
CONTROL_YEAR = ("1990-05", "1991-04")   # stability test: this year's forcing (RYF 1990-91), repeated ...
CONTROL_YEARS = 20                      # ... this many times (was 50: the first physically coupled runs left
                                        # ACCESS-OM2's range by year 14, so 20 years show it, in less than half the time)

# ==================================== Rollouts ======================================
N_PRIOR = 2         # the model sees 2 months (t-1 and t). The code below assumes 2.
WINDOW = 60         # months of truth every training sample needs after its start (48 free + 12 scored)
VALID_STEPS = 36    # validation: run 36 months from the true state and score all of them

# ===================================== Model ========================================
WIDTH_MULTIPLIER = 4   # UNet channels: 4 x 16 inputs = 64 at 1/2 resolution, 128 at 1/4, 256 in the bottleneck
SPECTRAL_WEIGHT = 2.0  # weight of the spectral loss (the main loss has weight 1)
# Spectral loss: along every grid row, the Arctic tripole rows included, with each row's true length
# (from GRID_PATH) setting its physical wavenumbers.
SPECTRAL_BANDS_PER_DECADE = 8          # wavenumber bands per factor of 10 in wavenumber

# ==================================== Training ======================================
# A rollout schedule says how long the training rollouts are, by epoch:
#   {first epoch: (free months without gradient, scored months)}
# e.g. {0: (0, 1), 15: (0, 4)} = epochs 0-14 score 1 month, from epoch 15 on score 4 months.
STAGE1_SCHEDULE = {0: (0, 1), 15: (0, 4), 45: (0, 12)}
STAGE1_EPOCHS = 90
STAGE1_LR = 1e-4        # learning rate (Adam), decayed to 0 over the stage by a cosine schedule
STAGE1_BATCH = 16       # samples per batch

# Stage 2 starts with 12 scored months and no free months, then lets the model run freely for 24, later 48,
# months before scoring 12.
STAGE2_SCHEDULE = {0: (0, 12), 5: (24, 12), 15: (48, 12)}   # epochs 0-4: score 12 months; 5-14: 24 free, then 12 scored; 15-: 48, then 12
STAGE2_EPOCHS = 30     # was 90: in intercomparison_90ep_v3 the stage 2 validation loss stopped improving early
STAGE2_LR = 5e-5
# Input noise in stage 2: every month of a training rollout, the model sees its input state plus Gaussian
# noise of this std (z-score units, white in space, ocean only), and is still scored against the truth.
# It learns to pull perturbed states back instead of amplifying them: without it, a grid-scale pattern
# at the tripole fold grew ~30% a year and the 50-year control blew up. A first guess, not tuned.
STAGE2_INPUT_NOISE = 0.1
STAGE2_BATCH = 8         # 8 samples x 4 members = 32 rollouts per batch

# ==================================== Ensemble ======================================
NOISE_CHANNELS = 4   # random-noise input channels
TRAIN_MEMBERS = 4    # rollouts per sample in training (CRPS needs at least 2)
EVAL_MEMBERS = 10    # members in the skill test

# Random seed: initial weights, order of the training samples, noise. A batch job sets it through the
# environment (qsub -v SEED=43 jobs/run_notebook.pbs); each seed gets its own run and figure folders.
SEED = int(os.environ.get("SEED", 42))

# ===================================== Outputs ======================================
RUN_DIR = REPO / "runs" / f"unet_crps_column_budget_seed{SEED}"       # trained weights and training logs, one folder per stage
FIG_DIR = REPO / "figures" / f"unet_crps_column_budget_seed{SEED}"    # figures
FIG_DIR.mkdir(parents=True, exist_ok=True)

# The normalised data, built once (a few minutes) and shared by both UNet notebooks.
# v5: the forcing is divided by one scale per flux (section 3). v6: the surface current in the state (data v5).
FIELDS_FILE = Path(DATA_PATH).parent / "emulator_cache" / "unet_notebooks_fields_v6.pt"

## 3. Data

Every variable is turned into a **z-score**: `z = (x - mean) / scale`. The mean is that grid cell's value for
that calendar month over 1970–2000 (the "climatology"), so z = 0 is the climatology. The scale:

- **State variables** (OHC, freshwater content, SST, SSS): that cell's std for that calendar month, so z = 1 is a
  typical anomaly there.
- **Forcing** (the surface fluxes): **one number per flux** for the whole ocean (`forcing_scale`: the typical size
  of the per-cell std). Some cells' fluxes barely vary (e.g. under sea ice). Dividing by their tiny std turned a
  small error of the coupler into z-scores of up to ~170 there, far outside anything the network had seen.

Land is set to 0.

The whole record is normalised once and kept in memory as tensors with shape `(time, variable, lat, lon)`.
A training sample is then just its start month: its inputs and targets are slices of these tensors.

We also keep, for the budget constraint and for converting predictions back to physical units:
the climatological mean and std (and the flux climatology and scales, for the coupler), the cell areas, the month lengths, and the ocean totals of every
forcing flux and of the climatological content.

The first cell below is the copied code that computes the climatology and the flux scales; the second uses it.

In [ ]:
# ==================================================================================================
# IDENTICAL CODE EXISTS IN src/Experiment/data.py
#   compute_normalisation, forcing_scale
# Copied verbatim so the notebook shows everything it runs. Editing it here does NOT change src/
# (which the test suite checks), and vice versa.
# ==================================================================================================

def compute_normalisation(ds, variables, fit_period, mask_variable):
    """
    The monthly climatology of every variable: the mean and std of each grid
    cell for each calendar month, fitted over fit_period (first and last month,
    inclusive). Both are returned on ds's whole time axis: every month gets its
    calendar month's values. Also returns the ocean mask: 1 where
    mask_variable has data in the first month, 0 on land.

    ds has dimensions (time, latitude, longitude). The numbers are the same as
    the "Spatial_climatology" strategy of Data.build_normalisation (the
    PyEarthTools version used by the older notebooks); the test suite checks
    this.
    """
    mask = xr.where(np.isnan(ds[mask_variable].isel(time=0)), 0, 1)
    fit = ds[variables].where(mask == 1).sel(time=slice(*fit_period))

    # Mean and std of every cell for each calendar month, (month, latitude, longitude). The std is that of
    # the anomalies from the monthly mean.
    monthly_mean = fit.groupby("time.month").mean("time")
    monthly_std = (fit.groupby("time.month") - monthly_mean).groupby("time.month").std("time")

    # Give every month of ds its calendar month's values: (time, latitude, longitude).
    calendar_month = ds.time.dt.month
    mean = monthly_mean.sel(month=calendar_month).drop_vars("month")
    std = monthly_std.sel(month=calendar_month).drop_vars("month")
    return mean, std, mask


def forcing_scale(std, ocean_area):
    """
    ONE number per forcing variable, to divide its anomaly by instead of each cell's own std.

    It is the typical size of that cell-and-month std over the ocean: the root of its mean square,
    weighted by cell area, over all ocean cells and all months.

    Why not each cell's own std, as for the state variables? Some cells have a flux that barely varies
    (e.g. the heat flux under permanent sea ice). Their std is tiny, so any small error in a flux
    computed by the coupler becomes an enormous z-score there (up to z ~ 170 for heat),
    far outside anything the network saw in training. With one scale per flux, the same error is
    small everywhere.

    std        : Dataset of (time, latitude, longitude) per-cell std, from compute_normalisation
    ocean_area : (latitude, longitude) DataArray, cell area on ocean and 0 on land
    returns    : {variable: float}
    """
    weight = ocean_area / ocean_area.sum()
    return {
        v: float(np.sqrt(((std[v].fillna(0.0) ** 2) * weight).sum(("latitude", "longitude")).mean("time")))
        for v in std.data_vars
    }


In [ ]:
if FIELDS_FILE.exists():
    # Built before: just load it.
    print(f"Loading the normalised data from {FIELDS_FILE}")
    saved = torch.load(FIELDS_FILE, weights_only=False)

else:
    print("Normalising the data (one pass over the file, a few minutes)...")

    # Open the file, rename the grid dimensions, and keep one value per month of TIME_AXIS.
    ds = xr.open_dataset(DATA_PATH).rename({"yt_ocean": "latitude", "xt_ocean": "longitude"})
    all_months = pd.date_range(TIME_AXIS[0], TIME_AXIS[1], freq="MS")
    ds = ds.sel(time=all_months, method="nearest")     # the file's time stamps are mid-month

    # The climatology: mean and std of every cell for every calendar month, fitted over NORM_FIT.
    # mean[v] and std[v] come back on the full time axis (each month gets its calendar month's values).
    # ocean_mask is 1 over ocean and 0 over land.
    mean, std, ocean_mask = compute_normalisation(ds, PROGNOSTIC + FORCING, NORM_FIT, mask_variable="total_surface_heat_flx")

    # Ocean mask and cell area (m^2, set to 0 on land so every sum over cells is a sum over the ocean).
    mask_array = ocean_mask.values.astype(np.float32)
    area_array = np.nan_to_num(ds["area_t"].isel(time=0).values.astype(np.float64)) * mask_array

    # The forcing is divided by ONE scale per flux, not each cell's std (see the text above): replace its std.
    flux_scales = forcing_scale(std[FORCING], xr.DataArray(area_array, dims=("latitude", "longitude")))
    for v in FORCING:
        std[v] = xr.full_like(std[v], flux_scales[v])
    print("one scale per flux:", flux_scales)

    # z-scores of every variable, (time, lat, lon) each. Land is NaN in the file: set to 0.
    z = ((ds[PROGNOSTIC + FORCING] - mean) / std).fillna(0.0)

    # Everything as tensors. np.stack(..., axis=1) puts the variables side by side: (time, variable, lat, lon).
    saved = {
        "state_z": torch.tensor(np.stack([z[v].values for v in PROGNOSTIC], axis=1), dtype=torch.float32),
        "forcing_z": torch.tensor(np.stack([z[v].values for v in FORCING], axis=1), dtype=torch.float32),
        # Climatological mean and std of the state variables, physical units.
        "state_mean": torch.tensor(np.stack([mean[v].fillna(0.0).values for v in PROGNOSTIC], axis=1), dtype=torch.float32),
        "state_std": torch.tensor(np.stack([std[v].fillna(0.0).values for v in PROGNOSTIC], axis=1), dtype=torch.float32),
        # Climatological mean of the fluxes (physical units) and their one scale each: the coupler needs both
        # to turn the fluxes it computes into the z-scores the network reads.
        "forcing_mean": torch.tensor(np.stack([mean[v].fillna(0.0).values for v in FORCING], axis=1), dtype=torch.float32),
        "forcing_scale": torch.tensor([flux_scales[v] for v in FORCING], dtype=torch.float32),
        "mask": torch.tensor(mask_array),
        "area": torch.tensor(area_array, dtype=torch.float64),
        # Ocean total of every forcing flux in every month (flux x area, summed over cells), e.g. W for the
        # heat flux: (time, forcing). Needed by the budget constraint.
        "forcing_total": torch.tensor(
            np.stack([(ds[v].fillna(0.0).values * area_array).sum(axis=(1, 2)) for v in FORCING], axis=1),
            dtype=torch.float64,
        ),
        # Length of every month in seconds (the budget constraint uses true month lengths).
        "month_seconds": torch.tensor([pd.Timestamp(t).days_in_month * 24 * 3600.0 for t in ds.time.values], dtype=torch.float64),
        "months": [pd.Timestamp(t).strftime("%Y-%m") for t in ds.time.values],
        "latitude": ds["latitude"].values,
        "longitude": ds["longitude"].values,
        "units": [ds[variable].attrs.get("units", "") for variable in PROGNOSTIC],
    }
    # Ocean total of the climatological content in every month, (time, state variable), e.g. J for OHC.
    saved["mean_total"] = (saved["state_mean"].double() * saved["area"]).sum(dim=(2, 3))

    FIELDS_FILE.parent.mkdir(parents=True, exist_ok=True)
    torch.save(saved, FIELDS_FILE)
    print(f"Saved {FIELDS_FILE}")

# Give everything a plain name.
state_z = saved["state_z"]              # (time, 6, lat, lon)  z-scores of OHC, freshwater content, SST, SSS, u_surf, v_surf
forcing_z = saved["forcing_z"]          # (time, 4, lat, lon)  z-scores of the 4 forcings
state_mean = saved["state_mean"]        # (time, 6, lat, lon)  climatological mean, physical units
state_std = saved["state_std"]          # (time, 6, lat, lon)  climatological std, physical units
forcing_mean = saved["forcing_mean"]    # (time, 4, lat, lon)  climatological mean of the 4 fluxes, physical units
flux_scale = saved["forcing_scale"]     # (4,)                 the one scale of each flux: forcing z = (flux - forcing_mean) / flux_scale
mask = saved["mask"]                    # (lat, lon)           1 = ocean, 0 = land
area = saved["area"]                    # (lat, lon)           cell area in m^2, 0 on land
months = saved["months"]                # ["1970-01", "1970-02", ...]
latitude = saved["latitude"]            # (lat,) nominal latitude of every grid row
longitude = saved["longitude"]          # (lon,)
units = saved["units"]                  # physical units of the 6 state variables

# Look up a month's position on the time axis: month_index["2005-02"] -> its time index.
month_index = {month: index for index, month in enumerate(months)}

# Training samples: every start month in TRAIN whose WINDOW following months also lie inside TRAIN,
# so training never sees a validation or test month.
train_start_months = torch.arange(month_index[TRAIN[0]], month_index[TRAIN[1]] + 1)
train_start_months = train_start_months[train_start_months + WINDOW <= month_index[TRAIN[1]]]

# Validation samples: every start month in VALID.
valid_start_months = torch.arange(month_index[VALID[0]], month_index[VALID[1]] + 1)

n_time, n_state, n_lat, n_lon = state_z.shape
print(f"{months[0]} .. {months[-1]}: {n_time} months on a {n_lat} x {n_lon} grid")
print(f"state  : {PROGNOSTIC}")
print(f"forcing: {FORCING}")
print(f"training samples: {len(train_start_months)} (start months {months[train_start_months[0]]} .. {months[train_start_months[-1]]})")
print(f"validation samples: {len(valid_start_months)}")

## 4. The hard global budget constraint

For column heat and freshwater content, the only way the **ocean total** can change is through the
surface. So between month a and month b:

$$\sum_\text{ocean} A\,(C_b - C_a) \;=\; \tfrac12\left(F_a L_a + F_b L_b\right)$$

where $C$ is the content per m², $A$ the cell area, $F$ the ocean-total surface flux and $L$ the month length.

After every prediction, `GlobalBudgetConstraint` computes how far the prediction is from this
(the residual $R$) and removes it by shifting the predicted z-score by the same amount at every ocean
cell. In physical units the correction is proportional to each cell's std, so it goes where the
content normally varies. The network still decides **where** heat and freshwater go; the constraint fixes
**how much** there is. So the ocean totals change only as the surface fluxes say: with ACCESS-OM2's fluxes they
follow ACCESS-OM2 exactly; with the coupler's, they drift only if the coupled fluxes do.

The fluxes come from the coupler, so every sample has its own, so `forward` can also be
given each sample's ocean-total fluxes (`flux_previous`, `flux_target`).

**Column by column.** The same budget holds in every column: its content changes by its own surface flux plus
the convergence of the ocean's lateral transport, which the 2-D content cannot show. So the flux part is not
left to the network: `ColumnBudget` adds each column's flux anomaly, integrated over the step, to its content,
and the network's own change in the content is taken as the transport. Left to itself, the network learned to
answer a flux with ~2.5 times the freshwater-content change ACCESS-OM2 shows (~10 times for heat), the more
so the longer its training rollouts; now a column takes in exactly what its surface flux gives it. It runs
first; `GlobalBudgetConstraint` then makes sure the transport moves heat and freshwater around without
changing the ocean total.

The first cell below is the copied constraints; the second builds it and checks it on a "nothing changes" forecast.

In [ ]:
# ==================================================================================================
# IDENTICAL CODE EXISTS IN src/Experiment/constraints.py
#   GlobalBudgetConstraint, ColumnBudget
# Copied verbatim so the notebook shows everything it runs. Editing it here does NOT change src/
# (which the test suite checks), and vice versa.
# ==================================================================================================

class GlobalBudgetConstraint(nn.Module):
    """
    forward(pred, previous, previous_index, target_index, flux_previous=None, flux_target=None)
        -> pred, with every enforced budget closed exactly.

    pred, previous   : (B, P, H, W) z-scores: the predicted next state and the
                       state it was predicted from
    previous_index,
    target_index     : (B,) time indices of those two states (their months)
    flux_previous,
    flux_target      : (B, C) ocean-integrated surface fluxes of those two months, one row per
                       sample, when the fluxes come from the coupler (each sample has
                       its own, computed from its own surface). Leave out to use ACCESS-OM2's
                       fluxes stored at construction (flux_global).

    For each budget, with content C = z * std + mean (physical, per cell):

        R = sum(area * (C_pred - C_previous)) - sign * (G_a L_a + G_b L_b) / 2

    where G is the ocean-integrated surface flux of a month. The correction
    adds k * std_target to C_pred with k = -R / sum(area * std_target): a
    uniform shift of the z-score over the ocean, so it is spread where the
    content varies and never divides by a small std. It is linear in pred, so
    gradients pass through it.
    """

    def __init__(self, budgets, content_std, mean_global, flux_global, month_seconds, area, mask):
        """
        budgets      : [(name, content_channel, flux_channel, flux_sign)]
        content_std  : {name: (T, H, W) std of the content, physical units}
        mean_global  : (T, P) ocean integral of the climatological mean of each prognostic variable
        flux_global  : (T, C) ocean integral of each forcing variable (physical)
        month_seconds: (T,) month lengths in seconds
        area, mask   : (H, W) cell area (m^2) and ocean mask
        """
        super().__init__()
        self.budgets = [(name, int(c), int(f), float(sign)) for name, c, f, sign in budgets]
        ocean_area = (area.double() * mask.double())
        # Non-persistent: rebuilt from the data, so they stay out of checkpoints.
        self.register_buffer("ocean_area", ocean_area, persistent=False)
        self.register_buffer("ocean", (mask > 0).float(), persistent=False)
        self.register_buffer("mean_global", mean_global.double(), persistent=False)
        self.register_buffer("flux_global", flux_global.double(), persistent=False)
        self.register_buffer("month_seconds", month_seconds.double(), persistent=False)
        for name, *_ in self.budgets:
            self.register_buffer(f"{name}_std", content_std[name].float(), persistent=False)

    def residual(self, pred, previous, previous_index, target_index, budget, flux_previous=None, flux_target=None):
        """(B,) budget residual R (content units x m^2) of one budget, and sum(area * std_target)."""
        name, c, f, sign = budget
        if flux_previous is None:   # ACCESS-OM2's fluxes of those months
            flux_previous = self.flux_global[previous_index]
            flux_target = self.flux_global[target_index]
        std = getattr(self, f"{name}_std")
        std_previous = std[previous_index].double()
        std_target = std[target_index].double()
        area = self.ocean_area
        change = ((pred[:, c].double() * std_target - previous[:, c].double() * std_previous) * area).sum(dim=(-2, -1))
        change = change + self.mean_global[target_index, c] - self.mean_global[previous_index, c]
        flux = sign * 0.5 * (flux_previous[:, f].double() * self.month_seconds[previous_index]
                             + flux_target[:, f].double() * self.month_seconds[target_index])
        return change - flux, (std_target * area).sum(dim=(-2, -1))

    def forward(self, pred, previous, previous_index, target_index, flux_previous=None, flux_target=None):
        out = pred.clone()
        for budget in self.budgets:
            residual, weight = self.residual(out, previous, previous_index, target_index, budget, flux_previous, flux_target)
            k = (residual / weight.clamp_min(1e-30)).to(pred.dtype)
            out[:, budget[1]] = out[:, budget[1]] - k[:, None, None] * self.ocean
        return out


class ColumnBudget(nn.Module):
    """
    forward(pred, previous, previous_index, target_index, forcing_previous, forcing_target)
        -> pred, with every budget's surface flux built into its content column by column.

    pred, previous    : (B, P, H, W) z-scores: the network's next state and the state it was predicted from
    previous_index,
    target_index      : (B,) time indices of those two states (their months)
    forcing_previous,
    forcing_target    : (B, C, H, W) the forcing inputs (z-scores: flux anomaly / scale) of those two months

    Every column's content C changes by its surface flux and by the convergence of the ocean's lateral
    transport D:

        C_b - C_a = sign * (F_a L_a + F_b L_b) / 2 + D

    The climatological fluxes and transport are in the climatology (C = mean + a), so in anomalies
    (a = z * std, F' = F - its climatology):

        a_b = a_a + sign * (F'_a L_a + F'_b L_b) / 2 + std_b * (pred - previous)

    The flux term is computed here; the network's change in z of the content channel is taken as the
    transport anomaly D / std_b. With no network change the anomaly persists and the flux anomaly goes
    straight into the column. Land (std 0) keeps the network's value. Run GlobalBudgetConstraint after it:
    D must not change the ocean total.
    """

    def __init__(self, budgets, content_std, forcing_scale, month_seconds):
        """
        budgets      : [(name, content_channel, flux_channel, flux_sign)], as GlobalBudgetConstraint
        content_std  : {name: (T, H, W) std of the content, physical units}
        forcing_scale: (C,) the one scale of each forcing: forcing z = (flux - climatology) / scale
        month_seconds: (T,) month lengths in seconds
        """
        super().__init__()
        self.budgets = [(name, int(c), int(f), float(sign)) for name, c, f, sign in budgets]
        # Non-persistent: rebuilt from the data, so they stay out of checkpoints.
        self.register_buffer("forcing_scale", torch.as_tensor(forcing_scale).float(), persistent=False)
        self.register_buffer("month_seconds", torch.as_tensor(month_seconds).float(), persistent=False)
        for name, *_ in self.budgets:
            self.register_buffer(f"{name}_std", content_std[name].float(), persistent=False)

    def forward(self, pred, previous, previous_index, target_index, forcing_previous, forcing_target):
        out = pred.clone()
        length_previous = self.month_seconds[previous_index][:, None, None]   # (B, 1, 1)
        length_target = self.month_seconds[target_index][:, None, None]
        for name, c, f, sign in self.budgets:
            std = getattr(self, f"{name}_std")
            std_previous, std_target = std[previous_index], std[target_index]   # (B, H, W)
            # The flux anomaly integrated between the two month centres, content units (e.g. J/m2).
            flux = sign * 0.5 * self.forcing_scale[f] * (forcing_previous[:, f].float() * length_previous
                                                         + forcing_target[:, f].float() * length_target)
            # What the column holds with no transport anomaly, as a z-score of the target month.
            column = (previous[:, c].float() * std_previous + flux) / std_target.clamp_min(1e-30)
            out[:, c] = pred[:, c] - previous[:, c] + torch.where(std_target > 0, column, previous[:, c].float())
        return out

In [ ]:
# Channel positions: which state channel is which content, which forcing channel is which flux.
ohc_channel = PROGNOSTIC.index("ocean_heat_content_2d")              # 0
freshwater_channel = PROGNOSTIC.index("ocean_freshwater_content_2d")  # 1
heat_flux_channel = FORCING.index("total_surface_heat_flx")           # 0
freshwater_flux_channel = FORCING.index("ocean_freshwater_flux")      # 1

constraint = GlobalBudgetConstraint(
    # One entry per budget: (name, content channel, flux channel, sign). Sign +1: positive flux = into the ocean.
    budgets=[
        ("heat", ohc_channel, heat_flux_channel, 1.0),
        ("freshwater", freshwater_channel, freshwater_flux_channel, 1.0),
    ],
    content_std={"heat": state_std[:, ohc_channel], "freshwater": state_std[:, freshwater_channel]},
    mean_global=saved["mean_total"],         # ocean total of the climatological content, every month
    flux_global=saved["forcing_total"],      # ocean total of every forcing flux, every month
    month_seconds=saved["month_seconds"],    # month lengths
    area=area,
    mask=mask,
)

# The column budget: the same budgets, each column taking in its own surface flux (the forcing input x its scale).
column_budget = ColumnBudget(
    budgets=constraint.budgets,
    content_std={"heat": state_std[:, ohc_channel], "freshwater": state_std[:, freshwater_channel]},
    forcing_scale=flux_scale,                # forcing z = (flux - climatology) / scale
    month_seconds=saved["month_seconds"],
)

# Check: forecast March 2005 as "nothing changes" (same z-scores as February), then apply the constraint.
# The residual is (change in ocean-total content) - (what the surface flux put in), so it should be ~0 after.
february = month_index["2005-02"]
previous_state = state_z[february : february + 1]          # (1, 4, lat, lon), the true February 2005
forecast = previous_state.clone()                          # "nothing changes"
previous_month = torch.tensor([february])
target_month = torch.tensor([february + 1])
constrained_forecast = constraint(forecast, previous_state, previous_month, target_month)

for budget in constraint.budgets:
    residual_before, _ = constraint.residual(forecast, previous_state, previous_month, target_month, budget)
    residual_after, _ = constraint.residual(constrained_forecast, previous_state, previous_month, target_month, budget)
    print(f"{budget[0]:>10} budget residual: {residual_before.item():+.3e} before, {residual_after.item():+.3e} after")

## 5. The coupler

In ACCESS-OM2 the surface fluxes depend on the ocean surface: a warm SST loses more heat to the air and cools back
down (the **air-sea feedback**). An emulator forced with ACCESS-OM2's own fluxes never feels that, so if its SST
drifts warm, nothing pulls it back. Here the fluxes are computed **the way ACCESS-OM2's coupler computes them**,
from the emulator's own surface.

What ACCESS-OM2 does (CICE's `gfdl_ocean_fluxes`, the GFDL/NCAR Large–Yeager bulk formulae), every 3 hours:
from the JRA55-do air temperature, humidity, wind and pressure (remapped to the ocean grid with the model's own
weights), the **SST** and the **surface current** (the stress and turbulent fluxes use the wind *relative* to the
current), it computes the sensible and latent heat, the evaporation, the upward longwave and the wind stress over
open water, weighted by 1 − sea-ice concentration. MOM also **restores the sea-surface salinity** towards a monthly
climatology (WOA13): a salt flux proportional to (target − SSS) and to the mass of the top cell, over 21.28 days,
everywhere (under the ice too), with its global mean removed, so the ocean-total salt is unchanged. Everything else
the ocean receives does not depend on the ocean surface: downward radiation, rain, snow, runoff, the sea-ice terms
and frazil.

So the coupler computes, for every month,

    flux = ACCESS-OM2's flux  −  B(ACCESS-OM2's surface)  +  B(the emulator's surface)

where **B** is the part that depends on the surface: the open-ocean fluxes, ported line by line from CICE and evaluated on **every 3-hourly JRA55-do
record** of the month (monthly-mean weather gets the latent heat wrong by ~25 W/m², the formulae are that
nonlinear). On ACCESS-OM2's own surface the coupler gives exactly ACCESS-OM2's fluxes; when the emulator's SST or
currents differ, the fluxes respond exactly as ACCESS-OM2's bulk formulae would. Offline check (2010, MOM's daily
SST and currents): the port reproduces MOM's open-ocean sensible, latent and longwave heat and evaporation with
R² = 1.0000. There is no neural network in it. The sea ice is **prescribed** (CICE's monthly concentration from the
same run).

The inputs are read straight from where they live, nothing is copied: JRA55-do from `/g/data/qv56`, the
remapping weights and the CICE output from `/g/data/ik11`. They are read once per run, for the months it needs
(~8 minutes for the 35 years of training), and kept in the node's memory (~170 MB a month, ~72 GB for training:
more than a 32 GB V100 holds); each batch's months are copied to the GPU when the coupler needs them.

**Lagged surface.** To predict month t+1 the emulator needs the fluxes of month t+1, but those depend on the
surface of month t+1, which it has not predicted yet. So B uses the surface **anomaly** of month t, laid on the
climatology of month t+1, for ACCESS-OM2's surface and the emulator's alike: the lags cancel when the emulator is
right.

**Salinity restoring.** B also holds MOM's salinity restoring, as the freshwater flux it equals (salt flux × −1000 / 35).
The data's freshwater flux leaves the restoring out (it is a salt flux in ACCESS-OM2), so it enters the forcing only as
R(the emulator's SSS) − R(ACCESS-OM2's SSS): if the emulator's SSS is saltier than ACCESS-OM2's, the coupler adds fresh
water, at MOM's rate. The target climatology cancels in that difference. Offline check: MOM's monthly restoring flux
against this formula, R² = 0.999 (1958). Without it the emulator's SSS and freshwater content had nothing pulling
them back and drifted in the control.

**In training** the gradients flow back through the bulk formulae, from the fluxes to the SST,
currents and SSS that made them, so the emulator learns how its own surface changes the fluxes it will feel.

The first cell below is the copied coupler code. The second builds the coupler for the training months; the
third checks it.

In [ ]:
# ==================================================================================================
# IDENTICAL CODE EXISTS IN src/Experiment/coupler.py
#   read_coupler_inputs, open_ocean_fluxes, monthly_open_ocean_fluxes, PhysicalCoupler, coupled_rollout
# Copied verbatim so the notebook shows everything it runs. Editing it here does NOT change src/
# (which the test suite checks), and vice versa.
# ==================================================================================================

import glob
import multiprocessing
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint as recompute_in_backward

# ---------------------------------------------------------------------------------------------------------
# Where the inputs live
# ---------------------------------------------------------------------------------------------------------
JRA55_DIR = "/g/data/qv56/replicas/input4MIPs/CMIP6/OMIP/MRI/MRI-JRA55-do-1-4-0/atmos/3hrPt"
REMAP_WEIGHTS = "/g/data/ik11/inputs/access-om2/input_20201102/common_1deg_jra55/rmp_jra55_cice_patch.nc"
ACCESS_OM2_RUN = "/g/data/ik11/outputs/access-om2/1deg_jra55_iaf_omip2_cycle6"

# The JRA55-do fields B needs, and how each is stored (half precision): (value - offset) * factor.
ATMOSPHERE = (
    ("tas", 273.15, 1.0),       # air temperature (K) -> degC
    ("huss", 0.0, 1000.0),      # specific humidity (kg/kg) -> g/kg
    ("uas", 0.0, 1.0),          # eastward wind (m/s)
    ("vas", 0.0, 1.0),          # northward wind (m/s)
    ("psl", 101325.0, 0.01),    # sea-level pressure (Pa) -> hPa from 1013.25
)
MAX_RECORDS = 31 * 8            # 3-hourly records in the longest month

# ---------------------------------------------------------------------------------------------------------
# Constants, as in cice5 drivers/auscom/ice_constants.F90 and GFDL surface_flux_mod
# ---------------------------------------------------------------------------------------------------------
RDGAS, RVGAS, CP_AIR, GRAV, VONKAR = 287.04, 461.50, 1005.0, 9.80616, 0.4
LVAP = 2.501e6                  # latent heat of vaporisation in CICE (J/kg)
HLV_MOM = 2.500e6               # MOM's: MOM's evaporation (kg/m2/s) = CICE's latent heat / HLV_MOM
SIGMA = 567.0e-10               # Stefan-Boltzmann (W/m2/K4)
TFFRESH = 273.15
D622 = RDGAS / RVGAS
D378 = 1.0 - D622
D608 = D378 / D622
KAPPA = RDGAS / CP_AIR
GUST = 1.0                      # m/s, added to the relative wind speed
Z_REF = 10.0                    # m, height of the atmospheric values
SALT_RESTORE_TSCALE = 21.28 * 86400.0   # s, MOM's salt_restore_tscale in ACCESS-OM2 (days)
S_REF = 35.0                    # psu, the data's reference salinity: freshwater flux = -1000 x salt flux / S_REF


# =========================================================================================================
# Reading the inputs (CPU, once per run)
# =========================================================================================================

def _remap_matrix(ocean_cells):
    """The SCRIP patch weights as a sparse matrix (ocean cells x JRA55 cells)."""
    import scipy.sparse as sp
    import xarray as xr
    w = xr.open_dataset(REMAP_WEIGHTS)
    matrix = sp.csr_matrix(
        (w.remap_matrix.values[:, 0], (w.dst_address.values - 1, w.src_address.values - 1)),
        shape=(w.sizes["dst_grid_size"], w.sizes["src_grid_size"]),
    )
    w.close()
    return matrix[ocean_cells]


def _jra55_file(field, year):
    files = glob.glob(f"{JRA55_DIR}/{field}/gr/*/{field}_*_gr_{year}01010000-*.nc")
    if len(files) != 1:
        raise FileNotFoundError(f"expected one JRA55-do {field} file for {year}, found {files}")
    return files[0]


_WORKER = {}


def _start_worker(ocean_cells):
    _WORKER["remap"] = _remap_matrix(ocean_cells)


def _read_year(job):
    """One year's wanted months: {"YYYY-MM": ((5, MAX_RECORDS, cells) float16, number of records)}."""
    import cftime
    import netCDF4
    year, wanted = job
    remap = _WORKER["remap"]
    out = {m: np.zeros((len(ATMOSPHERE), MAX_RECORDS, remap.shape[0]), dtype=np.float16) for m in wanted}
    n_records = {}
    for k, (field, offset, factor) in enumerate(ATMOSPHERE):
        with netCDF4.Dataset(_jra55_file(field, year)) as ds:
            t = ds["time"]
            dates = cftime.num2date(t[:], t.units, t.calendar)
            record_month = np.array([f"{d.year:04d}-{d.month:02d}" for d in dates])
            var = ds[field]
            var.set_auto_mask(False)
            for m in wanted:
                records = np.flatnonzero(record_month == m)
                assert len(records) and np.all(np.diff(records) == 1), (field, m)
                x = var[records[0] : records[-1] + 1].astype(np.float64).reshape(len(records), -1)
                remapped = np.asarray(remap @ x.T).T                    # (records, ocean cells)
                out[m][k, : len(records)] = ((remapped - offset) * factor).astype(np.float16)
                n_records[m] = len(records)
    return {m: (out[m], n_records[m]) for m in wanted}


def read_coupler_inputs(months, time_indices, mask, device, weather_device, processes=12):
    """
    Everything the coupler reads from outside the emulator, for the forcing months `months`.

    months       : ["YYYY-MM", ...] the forcing months to read (no repeats)
    time_indices : their positions on the emulator's time axis
    mask         : (H, W) 1 = ocean
    device       : where to keep the small inputs (the GPU)
    weather_device : where to keep the weather, ~170 MB a month: the GPU if it fits there (an 80-141 GB A100 or
                   H200), otherwise "cpu" (a 32 GB V100), and the coupler copies each batch's months to the GPU
    returns {
        "time_index": (M,) long, the months' positions on the emulator's time axis
        "atmosphere": (M, 5, MAX_RECORDS, cells) float16 on `weather_device`, every 3-hourly record of tas, huss,
                      uas, vas, psl remapped to the ocean cells as libaccessom2 does, stored as
                      (x - offset) * factor (see ATMOSPHERE); records past the end of a month are 0
        "n_records" : (M,) long on `device`, 3-hourly records in each month
        "open_water": (M, cells) float32 on `device`, 1 - CICE's monthly-mean sea-ice concentration
        "top_cell_mass": (M, cells) float32 on `device`, MOM's monthly-mean rho_dzt of the top cell (kg/m2)
    }
    """
    import xarray as xr
    months = list(months)
    assert len(set(months)) == len(months), "repeated months"
    ocean_cells = np.flatnonzero(np.asarray(mask).ravel() > 0)
    slot = {m: k for k, m in enumerate(months)}

    atmosphere = torch.zeros((len(months), len(ATMOSPHERE), MAX_RECORDS, len(ocean_cells)), dtype=torch.float16, device=weather_device)
    n_records = torch.zeros(len(months), dtype=torch.long)
    by_year = {}
    for m in months:
        by_year.setdefault(int(m[:4]), []).append(m)
    # Processes, not threads (HDF5 reads hold the GIL). Forked: the workers never touch the GPU.
    with multiprocessing.get_context("fork").Pool(processes, initializer=_start_worker, initargs=(ocean_cells,)) as pool:
        for done in pool.imap_unordered(_read_year, sorted(by_year.items())):
            for m, (values, n) in done.items():
                atmosphere[slot[m]] = torch.from_numpy(values).to(weather_device)
                n_records[slot[m]] = n

    # The month's sea-ice concentration, CICE monthly mean (iceh.YYYY-MM.nc of the run).
    ice_files = {Path(f).name[5:12]: f for f in glob.glob(f"{ACCESS_OM2_RUN}/output*/ice/OUTPUT/iceh.????-??.nc")}
    open_water = np.zeros((len(months), len(ocean_cells)), dtype=np.float32)
    for m in months:
        with xr.open_dataset(ice_files[m], decode_times=False) as ice:
            aice = np.nan_to_num(ice["aice_m"].values[0]).ravel()[ocean_cells]
        open_water[slot[m]] = 1.0 - np.clip(aice, 0.0, 1.0)

    # The mass of MOM's top cell (kg/m2, rho_dzt at the surface, monthly mean): the salinity restoring's flux is
    # proportional to it. From the run's ocean_month.nc, one file per year next to the ice files (records Jan..Dec).
    top_cell_mass = np.zeros_like(open_water)
    by_file = {}
    for m in months:
        by_file.setdefault(Path(ice_files[m]).parents[2] / "ocean" / "ocean_month.nc", []).append(m)
    for path, file_months in by_file.items():
        with xr.open_dataset(path, decode_times=False) as ocean:
            top = ocean["rho_dzt"].isel(st_ocean=0)
            for m in file_months:
                top_cell_mass[slot[m]] = np.nan_to_num(top.isel(time=int(m[5:7]) - 1).values).ravel()[ocean_cells]

    return {
        "time_index": torch.as_tensor(time_indices, dtype=torch.long),
        "atmosphere": atmosphere,
        "n_records": n_records.to(device),
        "open_water": torch.from_numpy(open_water).to(device),
        "top_cell_mass": torch.from_numpy(top_cell_mass).to(device),
    }


# =========================================================================================================
# The open-ocean bulk formulae (GPU, differentiable)
# =========================================================================================================

def saturation_vapour_pressure(t):
    """GFDL compute_es_k (Goff-Gratch; water above 0 C, ice below -20 C, linear blend between), Pa; t in K.
    (The model looks it up in a 0.1 K table of this function.)"""
    t_ice, t_boil = TFFRESH, TFFRESH + 100.0
    log_ice = (-9.09718 * (t_ice / t - 1.0) - 3.56654 * torch.log10(t_ice / t)
               + 0.876793 * (1.0 - t / t_ice) + np.log10(610.71))
    log_water = (-7.90298 * (t_boil / t - 1) + 5.02808 * torch.log10(t_boil / t)
                 - 1.3816e-07 * (10 ** ((1 - t / t_boil) * 11.344) - 1)
                 + 8.1328e-03 * (10 ** ((t_boil / t - 1) * (-3.49149)) - 1) + np.log10(101324.60))
    es_ice, es_water = 10.0 ** log_ice, 10.0 ** log_water
    blend = 0.05 * ((t_ice - t) * es_ice + (t - t_ice + 20.0) * es_water)
    return torch.where(t <= t_ice - 20.0, es_ice, torch.where(t >= t_ice, es_water, blend))


def salinity_restoring(sss, top_cell_mass, cell_area):
    """
    MOM5's sea-surface salinity restoring as ACCESS-OM2 runs it (ocean_sbc_nml: salt_restore_as_salt_flux,
    salt_restore_tscale = 21.28 days, zero_net_salt_restore, salt_restore_under_ice, no cap on the difference),
    as the freshwater flux it equals (kg/m2/s, into the ocean: -1000 x salt flux / S_REF, the data's convention),
    leaving out the terms without SSS (the target's), which cancel in the coupler's R(emulator) - R(ACCESS-OM2):
        MOM's salt flux = rho_dzt(top) (S_target - SSS) / (1000 tau), minus its ocean (area-weighted) mean
        R(SSS)          = rho_dzt(top) (SSS - S_REF) / (tau S_REF),     minus its ocean mean
    Offline check (1958, ACCESS-OM2's monthly sfc_salt_flux_restore against rho_dzt (S_target - sss)): R^2 = 0.999,
    tau = 21.1-21.4 days.

    sss (B, cells) psu; top_cell_mass (B, cells) kg/m2; cell_area (cells,) m2. Returns (B, cells).
    """
    restoring = top_cell_mass * (sss - S_REF) / (SALT_RESTORE_TSCALE * S_REF)   # S_REF: only to keep the numbers small
    weight = (cell_area / cell_area.sum()).to(restoring.dtype)
    return restoring - (restoring * weight).sum(dim=-1, keepdim=True)


def open_ocean_fluxes(tas, huss, uas, vas, psl, sst, u_ocean, v_ocean):
    """
    CICE's gfdl_ocean_fluxes -> surface_flux_1d (ncar_ocean_fluxes, corrected branch), for open water.

    tas (K), huss (kg/kg), uas, vas (m/s, east/north), psl (Pa), sst (K, as MOM writes it), u_ocean, v_ocean (m/s): tensors of
    broadcastable shapes, (B, records, cells) here. Returns (B, 4, records, cells), all positive INTO the ocean:
        heat        W/m2      -(sensible + latent + sigma SST^4)
        freshwater  kg/m2/s   -evaporation, as MOM receives it
        tau_x, tau_y  N/m2    stress on the ocean
    """
    # Air density and pressure at 10 m (cpl_forcing_handler.F90); the surface pressure is psl.
    tv0 = tas * (1.0 + D608 * huss)
    p_atm = psl - psl / (RDGAS * tv0) * GRAV * Z_REF
    t_surf = sst                                         # MOM's sst is in K
    e_sat = saturation_vapour_pressure(t_surf)
    q_sat = D622 * e_sat / (psl - D378 * e_sat)          # use_mixing_ratio = F, no 0.98 factor
    th_atm = tas * (psl / p_atm) ** KAPPA                # potential temperature
    u_rel = u_ocean - uas                                # relative wind (absolute_wind = F)
    v_rel = v_ocean - vas
    w_atm = torch.sqrt(u_rel ** 2 + v_rel ** 2 + GUST ** 2)

    # ncar_ocean_fluxes(u_del=w_atm, t=th_atm, ts=t_surf, q=huss, qs=q_sat, z=10), corrected branch
    t, ts, q, qs = th_atm, t_surf, huss, q_sat
    tv = t * (1 + 0.608 * q)
    u = torch.clamp(w_atm, min=0.5)
    u10 = u
    cd_n10 = (2.7 / u10 + 0.142 + 0.0764 * u10) / 1e3
    cd_n10_rt = torch.sqrt(cd_n10)
    ce_n10 = 34.6 * cd_n10_rt / 1e3
    stable = (t >= ts).to(t.dtype)                       # 0.5 + sign(0.5, t - ts): sign(0) = +
    ch_n10 = (18.0 * stable + 32.7 * (1 - stable)) * cd_n10_rt / 1e3
    cd, ch, ce = cd_n10, ch_n10, ce_n10
    for _ in range(2):
        cd_rt = torch.sqrt(cd)
        ustar = cd_rt * u
        tstar = (ch / cd_rt) * (t - ts)
        qstar = (ce / cd_rt) * (q - qs)
        bstar = GRAV * (tstar / tv + qstar / (q + 1 / 0.608))
        zeta = VONKAR * bstar * Z_REF / (ustar * ustar)
        zeta = torch.copysign(torch.clamp(zeta.abs(), max=10.0), zeta)
        # CICE: x2 = max(sqrt(abs(1 - 16 zeta)), 1). The same, with the max taken before the square root: at
        # zeta = 1/16 the square root's gradient is infinite, and 0 (from the max) x inf = NaN would poison
        # every weight's gradient.
        x2 = torch.sqrt(torch.clamp((1 - 16 * zeta).abs(), min=1.0))
        x = torch.sqrt(x2)
        is_stable = zeta > 0
        psi_m = torch.where(is_stable, -5 * zeta,
                            torch.log((1 + 2 * x + x2) * (1 + x2) / 8) - 2 * (torch.atan(x) - np.arctan(1.0)))
        psi_h = torch.where(is_stable, -5 * zeta, 2 * torch.log((1 + x2) / 2))
        u10 = u / (1 + cd_n10_rt * (np.log(Z_REF / 10) - psi_m) / VONKAR)
        cd_n10 = (2.7 / u10 + 0.142 + 0.0764 * u10) / 1e3
        cd_n10_rt = torch.sqrt(cd_n10)
        ce_n10 = 34.6 * cd_n10_rt / 1e3
        stable = (zeta >= 0).to(t.dtype)
        ch_n10 = (18.0 * stable + 32.7 * (1 - stable)) * cd_n10_rt / 1e3
        xx = (np.log(Z_REF / 10) - psi_m) / VONKAR
        cd = cd_n10 / (1 + cd_n10_rt * xx) ** 2
        xx = (np.log(Z_REF / 10) - psi_h) / VONKAR
        ch = ch_n10 / (1 + ch_n10 * xx / cd_n10_rt) * torch.sqrt(cd / cd_n10)
        ce = ce_n10 / (1 + ce_n10 * xx / cd_n10_rt) * torch.sqrt(cd / cd_n10)

    # Back in surface_flux_1d (rough_scale = 1)
    rho = p_atm / (RDGAS * tas * (1.0 + D608 * huss))
    sensible = CP_AIR * ch * w_atm * rho * (t_surf - th_atm)        # W/m2, upward
    evaporation = ce * w_atm * rho * (q_sat - huss)                 # kg/m2/s, upward
    heat = -(sensible + LVAP * evaporation + SIGMA * t_surf ** 4)
    freshwater = -evaporation * LVAP / HLV_MOM
    tau_x = -cd * w_atm * rho * u_rel
    tau_y = -cd * w_atm * rho * v_rel
    return torch.stack([heat, freshwater, tau_x, tau_y], dim=1)


def _records_sum(chunk, valid, sst, u_ocean, v_ocean):
    """open_ocean_fluxes over a chunk of 3-hourly records, summed over the records inside the month: (B, 4, cells).
    chunk (B, 5, records, cells) float16 as stored by read_coupler_inputs; valid (B, records) 1 inside the month."""
    tas, huss, uas, vas, psl = (chunk[:, k].float() / factor + offset for k, (_, offset, factor) in enumerate(ATMOSPHERE))
    fluxes = open_ocean_fluxes(tas, huss, uas, vas, psl, sst, u_ocean, v_ocean)   # (B, 4, records, cells)
    return (fluxes * valid[:, None, :, None]).sum(dim=2)


# The same, compiled for the GPU: torch.compile fuses the bulk formulae's many elementwise steps into a few kernels,
# so their intermediates never go through GPU memory (faster, and a fraction of the memory). On the CPU (the tests)
# it runs as written above.
_records_sum_compiled = torch.compile(_records_sum)


def monthly_open_ocean_fluxes(atmosphere, slots, n_records, open_water, surface, records_per_chunk=8):
    """
    The month-mean of open_ocean_fluxes over every 3-hourly record, times the open-water fraction.

    atmosphere : (M, 5, MAX_RECORDS, cells) float16, the weather of every month read, as stored by
                 read_coupler_inputs (on the GPU, or in host memory)
    slots      : (B,) which of those months each sample is in (samples may share a month)
    n_records  : (B,) records in each sample's month
    open_water : (B, cells)
    surface    : (B, 3, cells) SST (K), u_surf, v_surf (m/s), held over the month
    returns      (B, 4, cells) heat, freshwater, tau_x, tau_y into the ocean

    In chunks of records, each fetched from `atmosphere` (to the GPU) only when it is needed. With gradients nothing
    of a chunk is kept for the backward pass: it is fetched and computed again there. (Keeping it would hold every
    rollout month's weather on the GPU, ~170 MB per sample-month: more than any GPU has for a 60-month rollout.)
    """
    sst, u_ocean, v_ocean = (surface[:, k, None] for k in range(3))     # each (B, 1, cells)
    records_sum = _records_sum_compiled if surface.is_cuda else _records_sum
    # Each distinct month is fetched once (ensemble members share their month), then handed to
    # every sample in it.
    distinct, sample_month = slots.unique(return_inverse=True)
    distinct = distinct.to(atmosphere.device)

    def chunk_sum(first, sst, u_ocean, v_ocean):
        chunk = atmosphere[:, :, first : first + records_per_chunk][distinct].to(sst.device)[sample_month]
        record = torch.arange(first, first + chunk.shape[2], device=sst.device)
        valid = (record[None, :] < n_records[:, None]).float()                         # (B, records)
        return records_sum(chunk, valid, sst, u_ocean, v_ocean)

    total = 0.0
    with torch.autocast(device_type=surface.device.type, enabled=False):
        for first in range(0, int(n_records.max()), records_per_chunk):
            if torch.is_grad_enabled() and surface.requires_grad:
                total = total + recompute_in_backward(chunk_sum, first, sst, u_ocean, v_ocean, use_reentrant=False)
            else:
                total = total + chunk_sum(first, sst, u_ocean, v_ocean)
    return total / n_records[:, None, None] * open_water[:, None]


# =========================================================================================================
# The coupler
# =========================================================================================================

class PhysicalCoupler(nn.Module):
    """
    coupler(latest_state, months) -> (forcing_z, flux_totals)

        latest_state : (B, P, H, W) the emulator's latest state (z-scores), month t
        months       : (B,) time index of the month to compute the fluxes for, t+1 (one of the months read
                       by read_coupler_inputs)
        forcing_z    : (B, 4, H, W) those fluxes as the emulator's forcing input (z-scores)
        flux_totals  : (B, 4) float64 their ocean integrals (e.g. W), for the budget constraint

    Gradients flow through it (from the fluxes back to the SST and currents that made them), so in training the
    emulator learns how its own surface changes the fluxes it will feel next month.
    """

    def __init__(self, inputs, true_state_z, true_fluxes, surface_channels, state_mean, state_std,
                 forcing_mean, forcing_scale, mask, area):
        """
        inputs           : from read_coupler_inputs (the weather on the GPU or in host memory, the rest on the GPU)
        true_state_z     : (T, P, H, W) ACCESS-OM2's states (z-scores), the whole time axis
        true_fluxes      : (T, 4, H, W) ACCESS-OM2's fluxes in physical units, in the emulator's forcing order:
                           heat (W/m2), freshwater (kg/m2/s), tau_x, tau_y (N/m2)
        surface_channels : positions of sst, u_surf, v_surf, sss among the state variables
        state_mean,
        state_std        : (T, P, H, W) climatological mean and std of the state (to undo the z-scores)
        forcing_mean     : (T, 4, H, W) climatological mean of the fluxes, physical units
        forcing_scale    : (4,) the one scale of each flux (forcing z = (flux - mean) / scale)
        mask, area       : (H, W) ocean mask, cell area (m^2)
        """
        super().__init__()
        self.surface_channels = list(surface_channels)
        self.grid_shape = tuple(mask.shape)
        device = inputs["open_water"].device
        ocean = torch.nonzero(torch.as_tensor(mask).flatten() > 0).squeeze(1)
        at_ocean = lambda field: torch.as_tensor(field).flatten(-2)[..., ocean.cpu()].float()

        slot = torch.full((true_state_z.shape[0],), -1, dtype=torch.long)
        slot[inputs["time_index"]] = torch.arange(len(inputs["time_index"]))
        buffers = {
            "ocean_index": ocean,
            "slot": slot,                                                  # time index -> position in the inputs
            "n_records": inputs["n_records"],                              # (M,)
            "open_water": inputs["open_water"],                            # (M, cells)
            "top_cell_mass": inputs["top_cell_mass"],                      # (M, cells)
            "surface_mean": at_ocean(state_mean[:, self.surface_channels]),   # (T, 4, cells)
            "surface_std": at_ocean(state_std[:, self.surface_channels]),
            "forcing_mean": at_ocean(forcing_mean),                        # (T, 4, cells)
            "forcing_scale": torch.as_tensor(forcing_scale, dtype=torch.float32),
            "cell_area": at_ocean(area).double(),                          # (cells,)
            "mask": torch.as_tensor(mask).float(),
        }
        # persistent=False: rebuilt from the data, never written into checkpoints.
        for name, value in buffers.items():
            self.register_buffer(name, value.to(device), persistent=False)
        # The weather, (M, 5, MAX_RECORDS, cells) float16: a plain attribute, not a buffer, so it stays where
        # read_coupler_inputs put it when the coupler is moved, and is never written into checkpoints.
        self.atmosphere = inputs["atmosphere"]

        # The rest of every month's flux: ACCESS-OM2's flux minus B on ACCESS-OM2's own (lagged) surface.
        rest = []
        with torch.no_grad():
            for k, t in enumerate(inputs["time_index"].tolist()):
                z_true = at_ocean(true_state_z[t - 1, self.surface_channels])[None].to(device)
                b_true = self._surface_fluxes(z_true, torch.tensor([t], device=device), torch.tensor([k], device=device))
                rest.append(at_ocean(true_fluxes[t])[None].to(device) - b_true)
        self.register_buffer("rest", torch.cat(rest), persistent=False)    # (M, 4, cells)

    def _surface_fluxes(self, z_surface, months, slots):
        """B: the fluxes (B, 4, cells) of `months` that depend on the surface, from the surface z-scores (B, 4, cells)
        of the month before, laid on the climatology of `months`: the open-ocean fluxes (SST, currents) and, in the
        freshwater flux, the salinity restoring (SSS)."""
        surface = z_surface * self.surface_std[months] + self.surface_mean[months]     # K, m/s, m/s, psu
        open_ocean = monthly_open_ocean_fluxes(self.atmosphere, slots, self.n_records[slots], self.open_water[slots],
                                               surface[:, :3])
        restoring = salinity_restoring(surface[:, 3], self.top_cell_mass[slots], self.cell_area)
        return torch.cat([open_ocean[:, :1], open_ocean[:, 1:2] + restoring[:, None], open_ocean[:, 2:]], dim=1)

    def fluxes(self, latest_state, months):
        """(B, 4, cells) physical fluxes of `months` from the emulator's latest state (the month before)."""
        slots = self.slot[months]
        if (slots < 0).any():
            raise KeyError(f"the coupler has no inputs for time indices {months[slots < 0].tolist()}")
        # index_select, not [..., self.ocean_index]: the backward of plain indexing sorts the indices on the GPU
        # every step (slow); index_select's backward adds straight into place.
        z_surface = latest_state[:, self.surface_channels].flatten(2).index_select(-1, self.ocean_index).float()
        return self.rest[slots] + self._surface_fluxes(z_surface, months, slots)

    def forcing_z(self, fluxes, months):
        """(B, 4, H, W) the emulator's forcing input: (flux - climatology) / scale, 0 on land."""
        z = (fluxes - self.forcing_mean[months]) / self.forcing_scale[None, :, None]
        full = z.new_zeros(z.shape[0], z.shape[1], self.grid_shape[0] * self.grid_shape[1])
        full = full.index_copy(-1, self.ocean_index, z)        # (as above: no index sort)
        return full.view(z.shape[0], z.shape[1], *self.grid_shape)

    def ocean_totals(self, fluxes):
        """(B, 4) float64 ocean integral of each flux (flux x cell area, summed), e.g. W for the heat flux."""
        return (fluxes.double() * self.cell_area).sum(dim=-1)

    def forward(self, latest_state, months):
        fluxes = self.fluxes(latest_state, months)
        return self.forcing_z(fluxes, months), self.ocean_totals(fluxes)


@torch.no_grad()
def coupled_rollout(model, coupler, prior, target_indices, initial_index, column_budget, constraint, initial_forcing,
                    initial_flux_totals, members=1):
    """
    Free-running rollout like autoregressive_rollout, but every month's forcing comes from the coupler,
    computed from the emulator's own previous month.

    prior               (n_prior, P, H, W) the true initial states, oldest first
    target_indices      (N,)  time index of each predicted month (sets the atmosphere and sea ice used)
    initial_index       int   time index of the last prior state
    column_budget       the ColumnBudget: every column's content takes in the coupled flux
    constraint          the GlobalBudgetConstraint, closed with the coupled fluxes
    initial_forcing     (C, H, W) ACCESS-OM2's forcing input (z-scores) of the starting month, and
    initial_flux_totals (C,)  its ocean-total fluxes (the budgets of the first step average them with the
                              first coupled month's)
    returns  predictions (members, N, P, H, W) z-scores, and the ocean-total coupled fluxes
             (members, N, C) float64, both on the CPU
    """
    device = next(model.parameters()).device
    was_training = model.training
    model.eval()
    coupler, column_budget, constraint = coupler.to(device), column_budget.to(device), constraint.to(device)
    n_prognostic = prior.shape[1]
    state = prior.flatten(0, 1)[None].to(device).repeat(members, 1, 1, 1)
    previous = torch.full((members,), int(initial_index), device=device)
    forcing_previous = torch.as_tensor(initial_forcing, device=device).float()[None].repeat(members, 1, 1, 1)
    totals_previous = torch.as_tensor(initial_flux_totals, device=device).double()[None].repeat(members, 1)
    predictions, totals = [], []
    for target in target_indices:
        target = torch.full_like(previous, int(target))
        latest = state[:, -n_prognostic:]
        forcing, totals_target = coupler(latest, target)                     # fluxes from the emulator's own surface
        pred = model(state, forcing, coupler.mask)
        pred = column_budget(pred, latest, previous, target, forcing_previous, forcing)
        pred = constraint(pred, latest, previous, target, totals_previous, totals_target)
        predictions.append(pred.float().cpu())
        totals.append(totals_target.cpu())
        state = torch.cat([state[:, n_prognostic:], pred.to(state.dtype)], dim=1)
        previous, forcing_previous, totals_previous = target, forcing, totals_target
    model.train(was_training)
    return torch.stack(predictions, dim=1), torch.stack(totals, dim=1)

In [ ]:
# Positions of the surface variables the coupler reads among the state variables: SST and the current for the
# open-ocean fluxes, SSS for the salinity restoring.
surface_channels = [PROGNOSTIC.index(v) for v in ("sst", "u_surf", "v_surf", "sss")]

# ACCESS-OM2's own fluxes in physical units, (time, 4, lat, lon): back from their z-scores, flux = climatology + z x scale.
true_fluxes = forcing_z * flux_scale[None, :, None, None] + forcing_mean

# The months the coupler reads for training (both stages): every forcing month of the training and validation rollouts,
# from the month after the first training start to the 36th month after the last validation start (2004-12). The
# 1990-91 control year lies inside. Read straight from the JRA55-do and ACCESS-OM2 files (~8 minutes, 12 processes):
# ~170 MB a month, ~72 GB for these 418 months.
coupler_months = list(range(month_index[TRAIN[0]] + 1, month_index[VALID[1]] + VALID_STEPS + 1))

# Where to keep the weather: on the GPU if it fits with ~40 GB to spare for training (an 80 GB A100 or 141 GB H200),
# otherwise in the node's memory (a 32 GB V100), from where each batch's months are copied to the GPU (slower).
weather_bytes = len(coupler_months) * 5 * 248 * int(mask.sum()) * 2
gpu_bytes = torch.cuda.get_device_properties(0).total_memory if DEVICE.type == "cuda" else 0
WEATHER_DEVICE = DEVICE if gpu_bytes > weather_bytes + 40e9 else torch.device("cpu")
coupler_inputs = read_coupler_inputs([months[i] for i in coupler_months], coupler_months, mask, DEVICE, WEATHER_DEVICE)

coupler = PhysicalCoupler(
    coupler_inputs,
    true_state_z=state_z,                # ACCESS-OM2's surface: each flux is taken around it
    true_fluxes=true_fluxes,
    surface_channels=surface_channels,
    state_mean=state_mean,               # to turn the surface z-scores back into K, m/s and psu
    state_std=state_std,
    forcing_mean=forcing_mean,           # to turn its fluxes into the z-scores the network reads ...
    forcing_scale=flux_scale,            # ... (flux - forcing_mean) / flux_scale
    mask=mask,
    area=area,
)
print(f"coupler: {len(coupler_months)} months, {months[coupler_months[0]]} .. {months[coupler_months[-1]]}, "
      f"{coupler_inputs['atmosphere'].numel() * 2 / 1e9:.0f} GB of 3-hourly weather on {WEATHER_DEVICE}")


In [ ]:
# Checks over the 12 months of the control year, each month's fluxes from the true month before (lagged, as in use).
check_months = torch.arange(month_index[CONTROL_YEAR[0]] + 1, month_index[CONTROL_YEAR[1]] + 2)
is_ocean = mask > 0
cell_weight = (area[is_ocean] / area[is_ocean].sum()).float().to(DEVICE)   # each ocean cell's share of the ocean area
true_surface = state_z[check_months - 1].to(DEVICE)                       # ACCESS-OM2's states, z-scores

with torch.no_grad():
    # (a) On ACCESS-OM2's own surface the coupler must give ACCESS-OM2's own fluxes.
    on_truth = coupler.fluxes(true_surface, check_months.to(DEVICE))      # (12, 4, ocean cells)
    expected = true_fluxes[check_months][:, :, is_ocean].to(DEVICE)
    print("(a) on ACCESS-OM2's surface, largest |coupler - ACCESS-OM2|:")
    for c, name in enumerate(FORCING):
        print(f"    {name:24s} {(on_truth[:, c] - expected[:, c]).abs().max():.2e}   (its one scale: {flux_scale[c]:.3g})")

    # (b) The air-sea feedback: the same months with the SST 1 K warmer everywhere (in z-scores: + 1 K / std).
    sst_std = state_std[check_months, surface_channels[0]].to(DEVICE)
    warmer = true_surface.clone()
    warmer[:, surface_channels[0]] += torch.where(sst_std > 0, 1.0 / sst_std, torch.zeros_like(sst_std))
    change = coupler.fluxes(warmer, check_months.to(DEVICE)) - on_truth
    global_change = (change * cell_weight).sum(dim=-1).mean(dim=0)       # global mean, averaged over the 12 months
    print(f"(b) SST + 1 K: global-mean heat flux {global_change[0]:+.1f} W/m2, "
          f"freshwater flux {global_change[1] * 86400:+.3f} mm/day (more evaporation)")

    # (c) The relative wind: the surface current 0.1 m/s faster eastward everywhere.
    u_std = state_std[check_months, surface_channels[1]].to(DEVICE)
    faster = true_surface.clone()
    faster[:, surface_channels[1]] += torch.where(u_std > 0, 0.1 / u_std, torch.zeros_like(u_std))
    change = coupler.fluxes(faster, check_months.to(DEVICE)) - on_truth
    print(f"(c) u_surf + 0.1 m/s: global-mean tau_x {(change[:, 2] * cell_weight).sum(dim=-1).mean():+.4f} N/m2")


## 6. The model

`ForwardEmulator` wraps the `UNet`:

    next state = current state + UNet(state at t-1, state at t, forcing at t+1, noise)

- The UNet predicts the **change**, so "nothing changes" costs it nothing and it only learns the tendency.
- Inputs: 2 months × 6 state variables + 4 forcings = 16 channels, plus 4 noise channels in stage 2.
- The noise is fresh N(0, 1) at every call, so every call **samples** a next month. Run the same start
  10 times and you get a 10-member ensemble.
- Inside the UNet: two stride-2 downsamplings (to 1/4 resolution), and **partial convolutions**
  (Liu et al. 2018): each convolution sees only the ocean cells in its window (land is multiplied by 0) and
  its output is scaled up by (window size) / (number of ocean cells in it), so coastal cells are not damped.
  Before every convolution the field is padded to follow the grid: it wraps around in longitude and
  follows the tripole fold at the north edge.

The two copied cells below hold the UNet (with its partial convolutions), then `ForwardEmulator` and `load_trunk`
(which starts the stage 2 network from stage 1's weights). The third builds the stage 1 network.

In [ ]:
# ==================================================================================================
# IDENTICAL CODE EXISTS IN src/Emulator/om2_model_utils.py
#   PartialConv2d, upsample_on_grid, UNet
# Copied verbatim so the notebook shows everything it runs. Editing it here does NOT change src/
# (which the test suite checks), and vice versa.
# ==================================================================================================

class PartialConv2d(nn.Module):
    """
    A convolution that sees only ocean cells: a partial convolution (Liu et al. 2018,
    "Image Inpainting for Irregular Holes Using Partial Convolutions").

    forward(x, mask) -> (out, new_mask)
        x        : (B, C_in, H, W)
        mask     : (B, 1, H, W), or (1, 1, H, W) shared by the whole batch; 1 = ocean, 0 = land
        out      : (B, C_out, H_out, W_out)
        new_mask : 1 wherever the convolution window held at least one ocean cell

    Three steps:
    1. Pad x and the mask with ``padding`` cells on every side, following the ACCESS-OM2 tripolar grid:
         * east/west: longitude is periodic, so the columns wrap around. (With non-periodic padding the
           seam acted as a coastline neither side could see across; in the autoregressive OHC emulator
           98% of the fastest-growing perturbation sat at that seam.)
         * north: the tripole fold. Top-row cell i neighbours top-row cell nx - 1 - i, so the ghost rows
           above the top row are the top rows reversed in x and in y.
         * south (Antarctica): rows of zeros, marked as land, so the edge acts as a coastline.
    2. Convolve x * mask, i.e. with land set to 0.
    3. Multiply by (cells in the window) / (ocean cells in the window), so values next to a coast are
       not damped by the land around them. 0 where the window holds no ocean cell.
    """

    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.padding = padding
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=kernel_size, stride=stride)
        # Counts the ocean cells in every window: all-ones weights, never trained.
        self.mask_conv = nn.Conv2d(1, 1, kernel_size=kernel_size, stride=stride, bias=False)
        self.mask_conv.weight.data[:] = 1.0
        self.mask_conv.requires_grad_(False)
        self.kernel_area = kernel_size * kernel_size

    def forward(self, x, mask):
        p = self.padding
        # 1. Padding. Along y: [p zero rows (south), field, fold rows (north)], the fold rows being the top
        # p rows flipped in y and x (rows run south to north). Along x: [last p columns, field, first p
        # columns], i.e. longitude wraps around. Built with torch.cat only: the same values as
        # F.pad(mode="circular"), which copies with a slow strided kernel (training step 9% faster).
        x_fold = torch.flip(x[..., -p:, :], dims=(-2, -1))
        x = torch.cat([x.new_zeros(*x.shape[:-2], p, x.shape[-1]), x, x_fold], dim=-2)
        x = torch.cat([x[..., -p:], x, x[..., :p]], dim=-1)
        mask_fold = torch.flip(mask[..., -p:, :], dims=(-2, -1))
        mask = torch.cat([mask.new_zeros(*mask.shape[:-2], p, mask.shape[-1]), mask, mask_fold], dim=-2)
        mask = torch.cat([mask[..., -p:], mask, mask[..., :p]], dim=-1)

        # 2 and 3. Convolve the ocean cells only, and scale up by (cells / ocean cells) in each window.
        ocean_cells = self.mask_conv(mask)
        scale = torch.where(ocean_cells > 0, self.kernel_area / (ocean_cells + 1e-6), torch.zeros_like(ocean_cells))
        new_mask = (ocean_cells > 0).to(mask.dtype)
        return self.conv(x * mask) * scale, new_mask


def upsample_on_grid(x, size):
    """
    Bilinear upsampling of x (B, C, h, w) to size = (H, W) on the ACCESS-OM2 tripolar grid, with the
    same neighbours as PartialConv2d:
      * longitude (the last dimension) is periodic: one column is wrapped around on each side, so the
        east and west edges interpolate across the seam. W must be a whole multiple of w.
      * north: the tripole fold. One ghost row, the top row reversed in x, is added above the top row,
        so the output rows nearest the fold interpolate towards their fold neighbours instead of
        copying the top row. (South is left as it was: the edge row is repeated.)
    The ghost rows and columns are cut off again afterwards. The interpolation is given the true scale
    factors, so every output cell samples the same input position as plain F.interpolate(x, size).
    """
    h, w = x.shape[-2:]
    scale_x = size[1] // w                  # e.g. 2: every input column becomes 2 output columns
    x = torch.cat([x, torch.flip(x[..., -1:, :], dims=(-1,))], dim=-2)   # north ghost row: the fold
    x = torch.cat([x[..., -1:], x, x[..., :1]], dim=-1)                  # wrap one column each side
    # Under autocast PyTorch runs every interpolation in 32-bit (bicubic needs it, pytorch #121324), converting
    # the 16-bit activations up and back down. Bilinear is fine in 16-bit, so autocast is off here and the
    # interpolation keeps the input's precision.
    with torch.autocast(device_type=x.device.type, enabled=False):
        x = F.interpolate(x, scale_factor=(size[0] / h, scale_x), mode="bilinear", align_corners=False,
                          recompute_scale_factor=False)
    return x[..., :size[0], scale_x:scale_x + size[1]]


class UNet(nn.Module):
    """
    Mask-aware U-Net: two stride-2 encoder levels, a bottleneck, and a decoder with skip connections at
    every resolution. Every convolution is a PartialConv2d, so land never enters the computation.

    forward(x, mask) -> (B, output_channel_count, H, W)
        x    : (B, input_channel_count, H, W) inputs (prior states, forcing, [noise])
        mask : (1, 1, H, W) or (B, 1, H, W) ocean mask, 1 = ocean

    Channel widths scale with the number of inputs, so adding predictors widens the network instead of
    squeezing them through a fixed bottleneck:
        level 1 (1/2 resolution)   : width_multiplier * width_channel_count   (32 for 8 inputs and 4)
        level 2 (1/4 resolution)   : 2 x level 1
        bottleneck (1/4 resolution): 4 x level 1
    width_channel_count defaults to input_channel_count. Setting it lets extra inputs (the noise
    channels of the CRPS ensemble) leave every layer but the first unchanged, so a trained network's
    weights still fit. width_multiplier must be >= 2 for the first layer to carry every signed input
    through its ReLU (each needs a +/- pair of channels).

    Design notes:
      * The bottleneck (enc3a-c) and the first decoder layer (dec1a-c) are each three 3x3 convolutions
        with a ReLU after each: together a 7x7 window (3 -> 5 -> 7), with 27 instead of 49 weights per
        channel pair and two extra non-linearities.
      * The output layer sees only the upsampled decoder features, not the raw input: fed the raw input
        as well, it learned a sharpening stencil that grew grid-scale noise by ~17% per step. The
        residual wrapper (next = state + output) carries the state's grid-scale content forward instead.
      * Both stride-2 layers are 4x4 with padding 1, so every coarse cell is centred between two finer
        cells and each coarse grid is symmetric about the tripole fold: coarse cell j's fold neighbour
        is cell n - 1 - j, which is what the fold padding assumes. With a 3x3 stride-2 layer the 1/4
        resolution cells sit on every other 1/2 resolution cell, whose fold neighbours are the cells in
        between, so all six bottleneck layers saw a neighbour one 1/2 resolution cell away from the true
        one across the fold (in x and y), and the bilinear upsampling assumed centres half a cell off.
    """

    def __init__(self,
                 input_channel_count=2,
                 output_channel_count=2,
                 width_multiplier=4,
                 width_channel_count=None):
        super().__init__()

        # Channel widths, scaled from the number of (physical) input channels.
        width1 = int(round(width_multiplier * (width_channel_count or input_channel_count)))  # 1/2 resolution
        width2 = 2 * width1                                                                     # 1/4 resolution
        latent_width = 4 * width1                                                               # bottleneck

        # ---------- Encoder ----------
        self.enc1 = PartialConv2d(input_channel_count, width1, kernel_size=4, stride=2, padding=1)
        self.enc2 = PartialConv2d(width1, width2, kernel_size=4, stride=2, padding=1)
        # Bottleneck: three 3x3 layers, the receptive field of one 7x7.
        self.enc3a = PartialConv2d(width2, width2, kernel_size=3, stride=1, padding=1)
        self.enc3b = PartialConv2d(width2, width2, kernel_size=3, stride=1, padding=1)
        self.enc3c = PartialConv2d(width2, latent_width, kernel_size=3, stride=1, padding=1)

        # ---------- Decoder ----------
        # After the first upsample: latent + skip from enc2 (three 3x3 layers, as a 7x7).
        self.dec1a = PartialConv2d(latent_width + width2, width2, kernel_size=3, stride=1, padding=1)
        self.dec1b = PartialConv2d(width2, width2, kernel_size=3, stride=1, padding=1)
        self.dec1c = PartialConv2d(width2, width2, kernel_size=3, stride=1, padding=1)

        # After the second upsample: dec1 + skip from enc1.
        self.dec2 = PartialConv2d(width2 + width1, width1, kernel_size=3, stride=1, padding=1)

        # Full resolution: dec2 upsampled.
        self.dec3 = PartialConv2d(width1, output_channel_count, kernel_size=3, stride=1, padding=1)

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x, mask):
        # The whole network on one batch. Every self.encN / self.decN(...) call runs one PartialConv2d
        # with that layer's own weights and returns (features, mask). Example widths: 8 inputs, width_multiplier 4.
        # ================================ Encoder ================================
        # PartialConv2d.forward, 4x4 stride 2: full -> 1/2 resolution, C_in -> 32 channels.
        x1, mask1 = self.enc1(x, mask)
        x1 = self.relu(x1)

        # PartialConv2d.forward, 4x4 stride 2: 1/2 -> 1/4 resolution, 32 -> 64 channels.
        x2, mask2 = self.enc2(x1, mask1)
        x2 = self.relu(x2)

        # Bottleneck, at 1/4 resolution: three 3x3 PartialConv2d.forward calls, a ReLU after each.
        latent, latent_mask = self.enc3a(x2, mask2)                  # 64 -> 64 channels
        latent = self.relu(latent)
        latent, latent_mask = self.enc3b(latent, latent_mask)        # 64 -> 64
        latent = self.relu(latent)
        latent, latent_mask = self.enc3c(latent, latent_mask)        # 64 -> 128
        latent = self.relu(latent)

        # ================================ Decoder ================================
        # x1 and x2 (the encoder features at 1/2 and 1/4 resolution) are the skip
        # connections: they are joined back in at their own resolution.

        # The bottleneck is already at x2's 1/4 resolution: upsample_on_grid
        # interpolates to x2's grid size, wrapping around in longitude and folding at the north edge.
        y = upsample_on_grid(latent, x2.shape[2:])
        y_mask = F.interpolate(latent_mask, size=mask2.shape[2:], mode="nearest")
        y = torch.cat([y, x2], dim=1)                                # skip from enc2: 128 + 64 = 192 channels

        # Three 3x3 PartialConv2d.forward calls at 1/4 resolution, a ReLU after each.
        y, y_mask = self.dec1a(y, y_mask)                            # 192 -> 64
        y = self.relu(y)
        y, y_mask = self.dec1b(y, y_mask)                            # 64 -> 64
        y = self.relu(y)
        y, y_mask = self.dec1c(y, y_mask)                            # 64 -> 64
        y = self.relu(y)

        # Upsample 1/4 -> 1/2 resolution and join the skip from enc1.
        y = upsample_on_grid(y, x1.shape[2:])
        y_mask = F.interpolate(y_mask, size=mask1.shape[2:], mode="nearest")
        y = torch.cat([y, x1], dim=1)                                # 64 + 32 = 96 channels

        # PartialConv2d.forward, 3x3 at 1/2 resolution.
        y, y_mask = self.dec2(y, y_mask)                             # 96 -> 32
        y = self.relu(y)

        # Upsample 1/2 -> full resolution (the input's exact size, odd grids included).
        y = upsample_on_grid(y, x.shape[2:])

        # PartialConv2d.forward, 3x3 at full resolution with the original ocean mask:
        # the output, one channel per predicted variable. No ReLU: the change can be negative.
        change, _ = self.dec3(y, mask)                               # 32 -> C_out
        return change

In [ ]:
# ==================================================================================================
# IDENTICAL CODE EXISTS IN src/Experiment/models.py
#   ForwardEmulator, load_trunk
# Copied verbatim so the notebook shows everything it runs. Editing it here does NOT change src/
# (which the test suite checks), and vice versa.
# ==================================================================================================

class ForwardEmulator(nn.Module):
    """
    One-step forward emulator: next state = most recent state + tendency_network(inputs).

    forward(prior, forcing, mask) -> (B, P, H, W) the next state, in float32 (so the losses stay in full
    precision when training with 16-bit mixed precision)
        prior   : (B, n_prior * P, H, W) prior states stacked oldest first
        forcing : (B, C, H, W) forcing of the month being predicted
        mask    : (H, W) ocean mask, 1 = ocean

    The tendency network (the UNet, or the Samudra ConvNeXt U-Net) predicts the CHANGE from the most
    recent state, so persisting the current anomaly costs it nothing and it only has to learn the
    tendency. Land values are ignored: the losses and the evaluation mask them out.

    noise_channels > 0 (the CRPS ensemble): the network also gets that many channels of fresh N(0, 1)
    noise at every call, after the forcing, so every call samples a different next state. With 0 no
    noise channel is added.
    """

    def __init__(self, tendency_network, n_prognostic, noise_channels=0):
        super().__init__()
        self.tendency_network = tendency_network
        self.n_prognostic = n_prognostic          # P: the most recent state is the last P channels of prior
        self.noise_channels = noise_channels

    def forward(self, prior, forcing, mask):
        noise = torch.randn(prior.shape[0], self.noise_channels, *prior.shape[-2:], device=prior.device, dtype=prior.dtype)
        # The inputs side by side, laid out channels-last in memory (NHWC): faster 16-bit convolutions on
        # tensor cores, same results.
        x = torch.cat([prior, forcing, noise], dim=1).contiguous(memory_format=torch.channels_last)
        # One (1, 1, H, W) land mask broadcast over the batch: every partial convolution then counts the
        # ocean cells once, not once per sample.
        mask = mask.to(device=x.device, dtype=x.dtype)[None, None]
        next_state = prior[:, -self.n_prognostic :] + self.tendency_network(x, mask)
        return next_state.float()


def load_trunk(model, state_dict):
    """
    Load trained weights (e.g. the deterministic stage 1 network) into model, which may have extra
    noise input channels. Every weight is copied into the model's weight of the same name. The one
    that does not fit, the first convolution of a CRPS-ensemble model (its noise channels come after
    the physical inputs), gets the trained weights in its first channels and zeros for the noise
    channels. The model then starts out exactly as the trained network, with the noise switched off
    until training turns it on.
    """
    own = model.state_dict()
    with torch.no_grad():
        for name, value in state_dict.items():
            if value.shape == own[name].shape:
                own[name].copy_(value)
            else:
                own[name].zero_()
                own[name][:, : value.shape[1]].copy_(value)   # input channels: physical first, then noise
    return model

In [ ]:
# Input channels: 2 months x 6 state variables + 4 forcings = 16.
n_inputs = N_PRIOR * len(PROGNOSTIC) + len(FORCING)

L.seed_everything(SEED)   # same initial weights every time

unet = UNet(
    input_channel_count=n_inputs,
    output_channel_count=len(PROGNOSTIC),   # it predicts the change in all 6 state variables
    width_multiplier=WIDTH_MULTIPLIER,
)
model = ForwardEmulator(unet, n_prognostic=len(PROGNOSTIC))   # next state = current state + UNet(...)
model = model.to(memory_format=torch.channels_last)   # weights channels-last, like the inputs: faster 16-bit convolutions

# torch.compile: merges the padding, masking and scaling around every convolution into a few GPU kernels
# instead of copying the activations several times per layer (training steps ~2.5x faster on a V100; the
# results differ only by 16-bit rounding). The first call with each new batch size compiles, ~1 min.
model.tendency_network.compile()

n_parameters = sum(parameter.numel() for parameter in model.parameters())
print(f"UNet parameters: {n_parameters:,}")

## 7. Loss functions

All three work in z-score units and average over ocean cells weighted by cell area
(so a polar cell counts less than an equatorial one).

**MSE** (stage 1 only): $(x - y)^2$ per cell. Its best prediction is the *expected* next month.

**CRPS** (stage 2), per cell, for members $x_1 \dots x_M$ and the truth $y$:

$$\text{CRPS} = \underbrace{\frac1M\sum_i |x_i - y|}_\text{skill: members near the truth} \;-\; \underbrace{\frac{1}{2M(M-1)}\sum_{i \ne j} |x_i - x_j|}_\text{spread: members apart}$$

The spread term has no target of its own; the skill term limits it. Averaged over many training
samples, the loss is lowest when the members are drawn from the true distribution of outcomes.
Toy example: the truth is +1 or −1 with equal chance. Four members at {0, 0, 0, 0} score 1.0; members at
{+1, +1, −1, −1} score 0.33. MSE would prefer the 0s.

**Spectral** (weight 2): for each member separately, take every latitude row from 65°S to 60°N, Fourier
transform it in longitude, and add up the energy in log-spaced bands of physical zonal wavenumber
(cycles/km). The loss is the mean squared difference of log10(energy) between prediction and truth.
It ignores *where* features are (phase), only *how energetic* each scale is. CRPS is pointwise, so this
is the only term that looks at spatial structure.

The first cell below is the copied loss code; the second picks the losses for each stage.

In [ ]:
# ==================================================================================================
# IDENTICAL CODE EXISTS IN src/Emulator/om2_loss_functions.py
#   field_loss, EARTH_RADIUS_KM, grid_row_length_km, zonal_wavenumber_bands, spectral_loss
# (src also defines SPECTRAL_BANDS_PER_DECADE here; in this notebook it is set in the settings cell at
# the top.)
# Copied verbatim so the notebook shows everything it runs. Editing it here does NOT change src/
# (which the test suite checks), and vice versa.
# ==================================================================================================

def field_loss(*, pred_t, target_t, mask, area, method="mse", n_members=1, **_):
    """
    Ocean-masked, AREA-WEIGHTED mean of a per-cell error for one rollout step,
    in z-score units. ``method`` chooses the per-cell error:

      "mse"   squared error (pred_t - target_t)^2.

      "crps"  fair CRPS of an ensemble. The batch holds ``n_members`` (from the
              rollout context) consecutive members of each sample: pred_t is
              (B * M, ...) and target_t repeats the truth for every member.
              Per cell, the fair (unbiased in M) CRPS is

                  mean_i |x_i - y|  -  sum_{i,j} |x_i - x_j| / (2 M (M - 1))

              (Ferro 2014; used for ensemble training by e.g. NeuralGCM and
              AIFS-CRPS). The first term pulls every member towards the truth,
              the second pushes members apart, so the loss is minimised by
              members drawn from the distribution of the truth, not by their
              mean: sharp members are not penalised for unpredictable detail as
              they are by MSE.

    The per-cell error is then averaged over the ocean, each cell weighted by
    its area (``area`` from the rollout context, m^2), like every evaluation
    metric, the per-variable RMSE logged in training, and the closure terms. On
    the ACCESS-OM2 1-degree grid cell areas vary 17x (latitudinal refinement
    near the equator, small cells near the poles): an unweighted per-cell mean
    gave the regions poleward of 60 degrees ~27% of the loss for ~10% of the
    ocean area.

    Each training setup chooses ``method`` once when it builds its list of
    losses (the loss cell of the UNet notebooks, LOSS_TERMS in
    src/Experiment/losses.py); rollout-context entries it does not use are
    absorbed by ``**_``. The loss weight is applied by the caller.
    """
    if method == "mse":
        error = (pred_t - target_t) ** 2                               # (B, P, H, W)
    elif method == "crps":
        m = int(n_members)
        if m < 2:
            raise ValueError("field_loss(method='crps') needs an ensemble: n_members >= 2 in the rollout context")
        b = pred_t.shape[0] // m
        members = pred_t.reshape(b, m, *pred_t.shape[1:])
        truth = target_t.reshape(b, m, *target_t.shape[1:])[:, 0]
        skill = (members - truth[:, None]).abs().mean(dim=1)
        spread = (members[:, :, None] - members[:, None]).abs().sum(dim=(1, 2)) / (m * (m - 1))
        error = skill - 0.5 * spread                                    # (B, P, H, W)
    else:
        raise ValueError(f"Unknown field_loss method {method!r}: use 'mse' or 'crps'")

    # Area-weighted mean over the ocean. Ocean mask (H, W), 1 = ocean, broadcast to error's (B, P, H, W).
    cell_weight = (mask * area).to(device=error.device, dtype=error.dtype).expand_as(error)
    return (error * cell_weight).sum() / cell_weight.sum().clamp_min(1e-12)


EARTH_RADIUS_KM = 6371.0


def grid_row_length_km(geolat, geolon):
    """
    Length (km) of every grid row, (H, W) cell-centre coordinates -> (H,).

    A row runs all the way round (longitude is periodic), so its length is the
    sum of the great-circle distances between neighbouring cell centres,
    including the last cell to the first. On a latitude-longitude row this is
    2 pi R cos(latitude). On the ACCESS-OM2 tripole grid north of ~65N it is
    not: a row there is a closed curve round the two grid poles, thousands of
    km long even at the top row (where 2 pi R cos(89.8 deg) would give 160 km).
    geolat and geolon must be defined over land too (the grid file, not a
    land-masked copy).
    """
    lat = np.deg2rad(np.asarray(geolat, dtype=float))
    lon = np.deg2rad(np.asarray(geolon, dtype=float))
    if not (np.isfinite(lat).all() and np.isfinite(lon).all()):
        raise ValueError("grid_row_length_km needs geolat and geolon at every cell, land included")
    lat_next, lon_next = np.roll(lat, -1, axis=-1), np.roll(lon, -1, axis=-1)
    # Haversine distance between each cell and its eastern neighbour.
    h = np.sin((lat_next - lat) / 2) ** 2 + np.cos(lat) * np.cos(lat_next) * np.sin((lon_next - lon) / 2) ** 2
    return (2 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(np.clip(h, 0.0, 1.0)))).sum(axis=-1)


def zonal_wavenumber_bands(row_length_km, n_lon):
    """
    Sort every (grid row, along-row harmonic) into bands of PHYSICAL wavenumber
    k (cycles/km), so a band is the same length scale in every row and at every
    resolution (1 deg or 1/10 deg). Every row is used, the tripole rows of the
    Arctic included: a grid row is periodic everywhere, and its true length
    (grid_row_length_km) gives its harmonics a physical wavenumber.

    Returns (band, power_weight, k_band):
      band          (rows * harmonics,) band number of each (row, harmonic),
                    flattened row by row;
      power_weight  (rows, harmonics): multiplies |rfft|^2 so that it sums to
                    the row's variance, times the row's length / 2 pi R (its
                    area share; cos(latitude) on a latitude-longitude row);
      k_band        (n_bands,) central wavenumber of each band (cycles/km).
    """
    row_length_km = np.asarray(row_length_km, dtype=float)              # (rows,)
    length_share = row_length_km / (2 * np.pi * EARTH_RADIUS_KM)        # cos(latitude) on a lat-lon row

    # Physical wavenumber of every (row, harmonic): harmonic n (n waves along the row) has
    # k = n / row length. rfftfreq(n_lon, d=1/n_lon) lists the harmonics rfft returns,
    # 0 .. n_lon // 2; harmonic 0 is the row mean, which is dropped.
    harmonics = np.fft.rfftfreq(n_lon, d=1.0 / n_lon)[1:]              # (harmonics,)
    k = harmonics[None, :] / row_length_km[:, None]                    # (rows, harmonics)

    # Bands: equal steps in log10(k), SPECTRAL_BANDS_PER_DECADE of them per factor of 10, counted from
    # the smallest k. Then renumber them 0, 1, 2, ..., skipping any band no (row, harmonic) falls in.
    log_k = np.log10(k)
    band = np.floor((log_k - log_k.min()) * SPECTRAL_BANDS_PER_DECADE).astype(int)
    used, band = np.unique(band.ravel(), return_inverse=True)
    k_band = 10.0 ** (log_k.min() + (used + 0.5) / SPECTRAL_BANDS_PER_DECADE)  # band centres

    # Parseval: sum_n |rfft_n|^2 / n_lon^2 is the row's variance, once every harmonic below
    # Nyquist is counted twice (rfft returns n but not its mirror image -n, which has the same
    # power). Times the row's length share, i.e. its share of the area.
    counted_twice = np.where(harmonics < n_lon / 2, 2.0, 1.0)
    power_weight = length_share[:, None] * counted_twice[None, :] / n_lon**2
    return band, power_weight, k_band


def spectral_loss(*, row_length_km, n_lon):
    """
    Return the spectral loss for a grid with these row lengths and n_lon cells
    per row: the mean squared difference in log10 ENERGY per physical
    wavenumber band, per sample and variable.

    Every grid row of the ocean-masked field (land = 0) is Fourier transformed
    along the row, which is periodic. The power of each harmonic is summed into
    log-spaced bands of physical wavenumber (zonal_wavenumber_bands), weighting
    rows by their length (their area share), so the band energies add up to the
    area-weighted along-row variance. The loss compares log energies band by
    band.

    Every row is used, the Arctic tripole rows included. Leaving them out (as
    the loss did before) left the grid-scale structure there unconstrained, and
    that is where the UNet controls blew up: a pattern 1-2 cells wide on the
    top rows, at the tripole fold, grew ~30% a year.

    Why this form. The skill-test spectra showed two failures over long
    rollouts: large scales losing energy (the MSE fit damps anomalies, i.e.
    blurring) and small scales gaining it (grid-scale noise growing). A loss on
    band energies penalises both, in either direction. It does not ask for the
    right phase, so an unpredictable eddy in the wrong place is not penalised
    as if it were a wrong large-scale pattern, and it does not suppress small
    scales that are really there -- the point at 1/10 deg. Logs make every band
    count, not just the energetic large scales. (The previous version compared
    the amplitude of every individual 2-D Fourier mode: noisy, and its FFT ran
    across the non-periodic north/south edges.)

    ``row_length_km``: (H,) the length of every grid row (grid_row_length_km),
    which sets its physical wavenumbers. ``n_lon``: cells per row.

    The returned loss is called with the whole rollout context
    (``loss(**context)``); the loss weight is applied by the caller.
    """
    # The grid's band layout (see zonal_wavenumber_bands), computed once. The tensors are
    # moved to the field's device at every call: a few hundred kB, negligible next to the network.
    band, power_weight, k_band = zonal_wavenumber_bands(row_length_km, n_lon)
    band = torch.as_tensor(band)
    power_weight = torch.as_tensor(power_weight, dtype=torch.float32)

    def log_band_energy_mse(pred_t, target_t, mask):
        device = pred_t.device
        # Prediction and truth side by side with land set to 0: (2, B, P, rows, n_lon).
        # In float32: half-precision FFTs need power-of-two sizes.
        fields = torch.stack([pred_t.float(), target_t.float()]) * mask.to(device, torch.float32)
        # Power of every along-row harmonic except the row mean, as its share of the area-weighted variance.
        power = torch.fft.rfft(fields, dim=-1).abs().square()[..., 1:] * power_weight.to(device)
        # Energy per wavenumber band: add up the power of every (row, harmonic) in it. (2, B, P, n_bands)
        energy = power.new_zeros(*power.shape[:-2], k_band.size).index_add(-1, band.to(device), power.flatten(-2))
        pred_energy, target_energy = energy
        # Floor relative to each sample's total energy, so a band with no ocean
        # (or a zero field) never gives log(0).
        floor = 1.0e-8 * target_energy.sum(-1, keepdim=True) + 1.0e-20
        return (torch.log10(pred_energy + floor) - torch.log10(target_energy + floor)).square().mean()

    def loss_fn(*, pred_t, target_t, mask, **_):
        # Gradient checkpointing: the FFTs are recomputed in the backward pass instead of stored
        # (same value and gradients, less memory). Without gradients this is a plain call.
        return checkpoint(log_band_energy_mse, pred_t, target_t, mask, use_reentrant=False)

    return loss_fn


In [ ]:
# Each loss is a function that the training loop calls at every scored month as
#   loss_function(pred_t=..., target_t=..., mask=..., area=..., n_members=...)
# and that returns one number. All of them take the same inputs, even the ones they do not use.


def mse(pred_t, target_t, mask, area, n_members):
    # Area-weighted mean squared error over the ocean (n_members is not used).
    return field_loss(pred_t=pred_t, target_t=target_t, mask=mask, area=area, method="mse")


def crps(pred_t, target_t, mask, area, n_members):
    # Fair CRPS of an ensemble of n_members, area-weighted over the ocean.
    return field_loss(pred_t=pred_t, target_t=target_t, mask=mask, area=area, n_members=n_members, method="crps")


# Log energy per wavenumber band. Built here, once, because it needs the grid: the length of every
# grid row (km, all the way round; from the grid file's cell-centre coordinates) and the cells per row.
with xr.open_dataset(GRID_PATH) as grid:
    row_length_km = grid_row_length_km(grid["geolat_t"].values, grid["geolon_t"].values)   # (lat,)
spectral = spectral_loss(row_length_km=row_length_km, n_lon=mask.shape[-1])

# The losses of each stage: {name: (weight, loss function)}. They are added up at every scored month.
STAGE1_LOSSES = {"mse": (1.0, mse), "spectral": (SPECTRAL_WEIGHT, spectral)}
STAGE2_LOSSES = {"crps": (1.0, crps), "spectral": (SPECTRAL_WEIGHT, spectral)}

## 8. Rollout training

A training batch is a set of start months. For each one:

1. start from the **true** states of the start month and the month before;
2. run `n_free` months on the model's own predictions **without gradients** ("pushforward": cheap,
   and it puts the model in the states it actually drifts into);
3. run `n_scored` more months **with gradients**, scoring every month against ACCESS-OM2.

Every prediction passes through the budget constraint before it is scored and fed back.

**Coupled** (both stages): the forcing of every month comes from the coupler: the atmosphere and sea ice of
that month and the rollout's own latest SST and surface current. Gradients flow back through the coupler too, so the model
learns how its own surface changes the fluxes it will feel.
With `members` > 1 every sample is rolled out that many times (different noise each time), as CRPS needs.

Training uses 16-bit mixed precision, the Adam optimiser and a cosine learning-rate decay.
Validation always runs `VALID_STEPS` = 36 months from the true state, so its loss is comparable between epochs.

The copied cell below resumes an interrupted stage and reads its training log; the next one defines the
training itself.

In [ ]:
# ==================================================================================================
# IDENTICAL CODE EXISTS IN src/Experiment/training.py
#   prepare_resume, read_metrics
# Copied verbatim so the notebook shows everything it runs. Editing it here does NOT change src/
# (which the test suite checks), and vice versa.
# ==================================================================================================

def prepare_resume(run_dir):
    """
    The checkpoint to resume run_dir's training from, or None if there is none
    yet. ModelCheckpoint (see build_trainer) overwrites checkpoints/last.ckpt
    after every epoch, with the weights, optimizer, learning-rate schedule and
    epoch number, so trainer.fit(..., ckpt_path=prepare_resume(run_dir))
    carries on from the last completed epoch (the rollout schedule follows the
    epoch number). When resuming, metrics.csv is renamed to metrics_part<N>.csv
    first, because the CSV logger replaces metrics.csv when training starts;
    read_metrics joins the parts.
    """
    checkpoint = Path(run_dir, "checkpoints", "last.ckpt")
    if not checkpoint.exists():
        return None
    # A run stopped before its first log write has a checkpoint but no metrics.csv yet.
    log = Path(run_dir, "metrics.csv")
    if log.exists():
        n_parts = len(list(Path(run_dir).glob("metrics_part*.csv")))
        log.rename(log.with_name(f"metrics_part{n_parts}.csv"))
    return checkpoint


def read_metrics(run_dir):
    """
    The CSV training log of run_dir as one row per epoch: metrics_part0.csv, metrics_part1.csv, ...
    (from earlier, interrupted runs; see prepare_resume) then metrics.csv, keeping the latest value of
    every column for each epoch.
    """
    run_dir = Path(run_dir)
    # The parts in number order (as text, part10 would sort before part2), then the current log.
    parts = sorted(run_dir.glob("metrics_part*.csv"), key=lambda path: int(path.stem[len("metrics_part"):]))
    frames = [pd.read_csv(path) for path in [*parts, run_dir / "metrics.csv"] if path.exists()]
    return pd.concat(frames, ignore_index=True).groupby("epoch", as_index=False).last()

In [ ]:
class RolloutTraining(L.LightningModule):
    # Lightning runs the training loop for us. Every epoch it calls training_step on every batch of
    # training start months and validation_step on every batch of validation start months, and it uses
    # configure_optimizers to update the weights. A batch is a 1-D tensor of start months (time indices).

    def __init__(self, model, losses, schedule, lr, epochs, members, input_noise, coupler):
        super().__init__()
        self.model = model             # the ForwardEmulator being trained
        self.losses = losses           # {name: (weight, loss function)}
        self.schedule = schedule       # {first epoch: (free months, scored months)}
        self.lr = lr                   # learning rate
        self.epochs = epochs           # length of the stage (for the cosine learning-rate decay)
        self.members = members         # rollouts per sample: 1 = deterministic, 4 = CRPS ensemble
        self.input_noise = input_noise # std of the noise added to the input state in training (z-scores)
        self.column_budget = column_budget   # the column budget from section 4
        self.constraint = constraint   # the global budget constraint from section 4
        self.coupler = coupler         # the coupler (section 5): every month's fluxes

        # Copies of the data held by this module, so that Lightning moves them to the GPU with it and a
        # batch can be cut out of them on the GPU. persistent=False: not written into checkpoints.
        self.register_buffer("state_z", state_z, persistent=False)
        self.register_buffer("mask", mask, persistent=False)
        self.register_buffer("area", area.float(), persistent=False)

    def rollout_loss(self, start_months, n_free, n_scored, input_noise):
        # Roll out every sample from its start month. Returns the loss averaged over the scored months,
        # and each loss term separately (for the training log). Every month the model sees its input state
        # plus Gaussian noise of std input_noise over the ocean (0 = none), and the budget constraint closes
        # the budgets from that same input.

        # Each sample is rolled out `members` times (different noise each time), the copies side by side:
        # start months [a, b] with 4 members -> [a, a, a, a, b, b, b, b]. With 1 member nothing changes.
        t = start_months.repeat_interleave(self.members)

        # The model's input state: the true months t-1 and t, stacked as channels
        #   [OHC(t-1), freshwater(t-1), OHC(t), freshwater(t)]   -> shape (batch, 4, lat, lon)
        state = torch.cat([self.state_z[t - 1], self.state_z[t]], dim=1)

        # The forcing of month t+1, and its ocean totals for the budget constraint:
        # computed by the coupler from the atmosphere and sea ice of month t+1 and the latest predicted
        #                   SST and current (month t), so the model feels the fluxes its own surface produces.
        def forcing_of_next_month(state, t):
            return self.coupler(state[:, n_state:], t + 1)

        # The budget averages the fluxes of the month before and the month predicted, so carry the latest totals
        # along. At the start: ACCESS-OM2's totals of the start month.
        totals_before = self.constraint.flux_global[t]
        # The column budget averages the flux fields the same way: ACCESS-OM2's forcing of the start month.
        forcing_before = forcing_z[t.cpu()].to(state.device)

        # ---- Free months: the model runs on its own predictions, with no gradients ----
        with torch.no_grad():
            for month in range(n_free):
                seen = state + input_noise * torch.randn_like(state) * self.mask          # the input, with noise
                forcing, totals = forcing_of_next_month(state, t)
                prediction = self.model(seen, forcing, self.mask)                         # month t+1, (batch, 4, lat, lon)
                prediction = self.column_budget(prediction, seen[:, n_state:], t, t + 1, forcing_before, forcing)   # each column takes in its flux
                prediction = self.constraint(prediction, seen[:, n_state:], t, t + 1, totals_before, totals)   # close the global budgets
                forcing_before, totals_before = forcing, totals
                state = torch.cat([state[:, n_state:], prediction], dim=1)                # drop month t-1, add month t+1
                t = t + 1

        # Under mixed precision PyTorch caches 16-bit copies of the weights. Copies made just now, without
        # gradients, would cut the scored months off from the weights, so throw them away.
        torch.clear_autocast_cache()

        # ---- Scored months: same steps, now with gradients, and every month compared with the truth ----
        # Each loss term, averaged over the scored months.
        loss_parts = {name: 0.0 for name in self.losses}
        for month in range(n_scored):
            # All 12 scored months keep their intermediate values for the backward pass: at most ~23 GiB
            # (CRPS, 32 rollouts), so it fits on a 32 GB V100 and is ~13% faster than recomputing them.
            seen = state + input_noise * torch.randn_like(state) * self.mask              # the input, with noise
            forcing, totals = forcing_of_next_month(state, t)                             # gradients flow through the coupler
            prediction = self.model(seen, forcing, self.mask)                             # month t+1
            prediction = self.column_budget(prediction, seen[:, n_state:], t, t + 1, forcing_before, forcing)   # each column takes in its flux
            prediction = self.constraint(prediction, seen[:, n_state:], t, t + 1, totals_before, totals)   # close the global budgets
            forcing_before, totals_before = forcing, totals

            # Score the prediction against ACCESS-OM2 in that month, with every loss term.
            for name, (weight, loss_function) in self.losses.items():
                loss_parts[name] = loss_parts[name] + weight * loss_function(
                    pred_t=prediction, target_t=self.state_z[t + 1], mask=self.mask, area=self.area, n_members=self.members
                ) / n_scored

            state = torch.cat([state[:, n_state:], prediction], dim=1)   # feed the prediction back in
            t = t + 1

        # The loss is the sum of the terms; the terms go to the training log (detached: for the log only).
        total_loss = sum(loss_parts.values())
        return total_loss, {name: value.detach() for name, value in loss_parts.items()}

    def training_step(self, start_months):
        # Rollout lengths for this epoch: the schedule entry with the latest first epoch already reached
        # (every schedule starts at epoch 0).
        latest_start = max(first_epoch for first_epoch in self.schedule if first_epoch <= self.current_epoch)
        n_free, n_scored = self.schedule[latest_start]

        loss, loss_parts = self.rollout_loss(start_months, n_free, n_scored, self.input_noise)

        # Under mixed precision a NaN loss silently skips the weight update, so the stage would "train"
        # on without changing anything. Stop instead.
        if not torch.isfinite(loss):
            raise FloatingPointError(f"Non-finite training loss in epoch {self.current_epoch}")

        # Write the epoch-mean losses to the training log (metrics.csv).
        self.log("train_loss", loss, prog_bar=True, on_step=False, on_epoch=True, batch_size=len(start_months))
        self.log_dict({f"train_{name}": value for name, value in loss_parts.items()},
                      on_step=False, on_epoch=True, batch_size=len(start_months))
        return loss

    def validation_step(self, start_months):
        # Validation: VALID_STEPS months from the true state, all scored, no free months, no input noise.
        loss, loss_parts = self.rollout_loss(start_months, 0, VALID_STEPS, 0.0)
        self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True, batch_size=len(start_months))
        self.log_dict({f"val_{name}": value for name, value in loss_parts.items()},
                      on_step=False, on_epoch=True, batch_size=len(start_months))

    def configure_optimizers(self):
        # Adam, with the learning rate decaying from lr to 0 along a cosine over the stage's epochs.
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.epochs)
        return {"optimizer": optimizer, "lr_scheduler": scheduler}

## 9. Train

In [ ]:
# ======================= Stage 1: deterministic warm-up (MSE + 2 x spectral), rollouts 1 -> 4 -> 12 months =======================
# If this stage was trained before, its weights are in RUN_DIR/shared/weights.pt: load them and skip training.
stage1_dir = RUN_DIR / "shared"
stage1_weights = stage1_dir / "weights.pt"

if stage1_weights.exists():
    model.load_state_dict(torch.load(stage1_weights, map_location="cpu"))
    print(f"Stage 1: loaded trained weights from {stage1_weights}")

else:
    L.seed_everything(SEED)   # same order of training samples (and same noise) every time

    # What to train, with which losses, and for how long.
    training = RolloutTraining(
        model,
        losses=STAGE1_LOSSES,
        schedule=STAGE1_SCHEDULE,
        lr=STAGE1_LR,
        epochs=STAGE1_EPOCHS,
        input_noise=0.0,           # stage 1: no input noise
        members=1,
        coupler=coupler,           # the fluxes come from the coupler
    )

    # The batches: start months, shuffled for training.
    # A batch is just B start months (time indices): the DataLoader batches the tensor of indices directly.
    train_loader = DataLoader(torch.as_tensor(train_start_months, dtype=torch.long), batch_size=STAGE1_BATCH, shuffle=True)
    valid_loader = DataLoader(torch.as_tensor(valid_start_months, dtype=torch.long), batch_size=STAGE1_BATCH)

    trainer = L.Trainer(
        max_epochs=STAGE1_EPOCHS,
        accelerator="gpu",
        devices=1,
        precision="16-mixed",          # 16-bit arithmetic where it is safe: faster, less memory
        benchmark=True,                # input sizes never change, so let cuDNN pick its fastest kernels once
        num_sanity_val_steps=0,        # skip Lightning's validation dry run before training
        enable_model_summary=False,
        logger=CSVLogger(save_dir=str(stage1_dir), name="", version=""),                 # the log: RUN_DIR/shared/metrics.csv
        callbacks=[ModelCheckpoint(dirpath=stage1_dir / "checkpoints", save_last=True, enable_version_counter=False)],  # always checkpoints/last.ckpt, overwritten every epoch
    )

    # If an earlier run of this stage was cut off (e.g. by a job's walltime), prepare_resume returns its
    # checkpoints/last.ckpt and training carries on from the last finished epoch; otherwise it returns None.
    trainer.fit(training, train_loader, valid_loader, ckpt_path=prepare_resume(stage1_dir))

    torch.save(model.state_dict(), stage1_weights)
    print(f"Stage 1: trained, saved {stage1_weights}")
    torch.cuda.empty_cache()   # give the GPU memory of the finished stage back

In [ ]:
# The stage 2 network: the same UNet, with NOISE_CHANNELS extra input channels for the noise.
L.seed_everything(SEED)
ensemble_unet = UNet(
    input_channel_count=n_inputs + NOISE_CHANNELS,
    output_channel_count=len(PROGNOSTIC),
    width_multiplier=WIDTH_MULTIPLIER,
    width_channel_count=n_inputs,   # channel widths follow the 8 physical inputs, so they match stage 1's
)
# noise_channels: draws fresh N(0, 1) noise for these channels at every call.
ensemble_model = ForwardEmulator(ensemble_unet, n_prognostic=len(PROGNOSTIC), noise_channels=NOISE_CHANNELS)
ensemble_model = ensemble_model.to(memory_format=torch.channels_last)
ensemble_model.tendency_network.compile()   # torch.compile, as for the stage 1 network

# Start from stage 1's weights. The weights of the new noise channels start at zero,
# so the ensemble model starts out exactly as the stage 1 model.
ensemble_model = load_trunk(ensemble_model, model.state_dict())

In [ ]:
# ======================= Stage 2: CRPS ensemble (CRPS + 2 x spectral), 4 members, scored on months 25-36, then 49-60 =======================
# If this stage was trained before, its weights are in RUN_DIR/crps_ensemble/weights.pt: load them and skip training.
stage2_dir = RUN_DIR / "crps_ensemble"
stage2_weights = stage2_dir / "weights.pt"

if stage2_weights.exists():
    ensemble_model.load_state_dict(torch.load(stage2_weights, map_location="cpu"))
    print(f"Stage 2: loaded trained weights from {stage2_weights}")

else:
    L.seed_everything(SEED)   # same order of training samples (and same noise) every time

    # What to train, with which losses, and for how long.
    training = RolloutTraining(
        ensemble_model,
        losses=STAGE2_LOSSES,
        schedule=STAGE2_SCHEDULE,
        lr=STAGE2_LR,
        epochs=STAGE2_EPOCHS,
        input_noise=STAGE2_INPUT_NOISE,
        members=TRAIN_MEMBERS,
        coupler=coupler,           # the fluxes come from the coupler
    )

    # The batches: start months, shuffled for training.
    # A batch is just B start months (time indices): the DataLoader batches the tensor of indices directly.
    train_loader = DataLoader(torch.as_tensor(train_start_months, dtype=torch.long), batch_size=STAGE2_BATCH, shuffle=True)
    valid_loader = DataLoader(torch.as_tensor(valid_start_months, dtype=torch.long), batch_size=STAGE2_BATCH)

    trainer = L.Trainer(
        max_epochs=STAGE2_EPOCHS,
        accelerator="gpu",
        devices=1,
        precision="16-mixed",          # 16-bit arithmetic where it is safe: faster, less memory
        benchmark=True,                # input sizes never change, so let cuDNN pick its fastest kernels once
        num_sanity_val_steps=0,        # skip Lightning's validation dry run before training
        enable_model_summary=False,
        logger=CSVLogger(save_dir=str(stage2_dir), name="", version=""),                 # the log: RUN_DIR/crps_ensemble/metrics.csv
        callbacks=[ModelCheckpoint(dirpath=stage2_dir / "checkpoints", save_last=True, enable_version_counter=False)],  # always checkpoints/last.ckpt, overwritten every epoch
    )

    # If an earlier run of this stage was cut off (e.g. by a job's walltime), prepare_resume returns its
    # checkpoints/last.ckpt and training carries on from the last finished epoch; otherwise it returns None.
    trainer.fit(training, train_loader, valid_loader, ckpt_path=prepare_resume(stage2_dir))

    torch.save(ensemble_model.state_dict(), stage2_weights)
    print(f"Stage 2: trained, saved {stage2_weights}")
    del training, trainer      # they hold the coupler: let its memory go when it is replaced for the tests
    torch.cuda.empty_cache()   # give the GPU memory of the finished stage back

In [ ]:
# Training curves: the loss per epoch, from each stage's metrics.csv.
# Validation always scores 36 months from the true state, so its loss is comparable across epochs.
fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
for ax, stage_folder in zip(axes, ["shared", "crps_ensemble"]):
    metrics = read_metrics(RUN_DIR / stage_folder)   # one row per epoch
    for column in ["train_loss", "val_loss"]:
        rows = metrics.dropna(subset=[column])
        ax.plot(rows["epoch"], rows[column], label=column, lw=2)
    ax.set_title(stage_folder)
    ax.set_xlabel("epoch")
    ax.grid(alpha=0.3)
    ax.legend(frameon=False)
fig.savefig(FIG_DIR / "training_curves.png", dpi=150, bbox_inches="tight")

## 10. Skill test: 2005–2018, coupled

Start from the true state of February 2005 and run freely to December 2018, **coupled**: the real atmosphere
and sea ice, and the fluxes from the coupler and the emulator's own SST and current
(166 months): 10 ensemble members (`EVAL_MEMBERS`), never shown the truth again. Results are converted back to physical units as
**anomalies** from the 1970–2000 monthly climatology (z × std).

Baselines: **persistence** (February 2005's anomaly held fixed) and **climatology** (zero anomaly).

The cell below runs it with `coupled_rollout` (copied in section 5).

In [ ]:
start = month_index[TEST_START]                       # the last true month the model is given
end = month_index[TEST_END] + 1                       # the last month it predicts (2018-12)
target_months = torch.arange(start + 1, end + 1)      # time index of every predicted month
dates = pd.to_datetime([months[i] for i in target_months.tolist()])   # their dates, for the plots

ensemble_model = ensemble_model.to(DEVICE)   # the rollout runs wherever the model is

# The coupler for the test months: free the training months' weather, read the test months'.
del coupler, coupler_inputs
torch.cuda.empty_cache()
test_inputs = read_coupler_inputs([months[i] for i in target_months.tolist()], target_months.tolist(), mask, DEVICE, WEATHER_DEVICE)
coupler = PhysicalCoupler(
    test_inputs, true_state_z=state_z, true_fluxes=true_fluxes, surface_channels=surface_channels,
    state_mean=state_mean, state_std=state_std, forcing_mean=forcing_mean, forcing_scale=flux_scale, mask=mask, area=area,
)

# Free run from the true states of January and February 2005, coupled: the real atmosphere and sea ice, the
# fluxes from the coupler and the emulator's own SST and current, and the budgets closed with those fluxes.
z_predicted, coupled_totals = coupled_rollout(
    ensemble_model,
    coupler,
    prior=state_z[start - 1 : start + 1],               # (2 months, 4, lat, lon): the true starting states
    target_indices=target_months,                       # every predicted month (sets the atmosphere and sea ice used)
    initial_index=start,                                # the starting month
    column_budget=column_budget,
    constraint=constraint,
    initial_forcing=forcing_z[start],                     # ACCESS-OM2's forcing input of the starting month
    initial_flux_totals=saved["forcing_total"][start],  # ACCESS-OM2's ocean-total fluxes of February 2005
    members=EVAL_MEMBERS,
)
# z_predicted: (members, months, 4, lat, lon) z-scores; coupled_totals: (members, months, 4) ocean-total fluxes. Both on the CPU.

# Back to physical units, as anomalies from the monthly climatology: anomaly = z x std.
std_of_targets = state_std[target_months]                         # (months, 4, lat, lon)
predicted_anomaly = z_predicted * std_of_targets                  # (members, months, 4, lat, lon)
true_anomaly = state_z[target_months] * std_of_targets            # (months, 4, lat, lon)
persistence_anomaly = state_z[start] * state_std[start]           # (2, lat, lon): February 2005, held fixed

n_members = predicted_anomaly.shape[0]
ocean = mask.numpy() == 1               # (lat, lon) True over the ocean, for the maps
area_weight = area / area.sum()         # (lat, lon) each ocean cell's share of the ocean area; sums to 1

print(f"Predicted {months[target_months[0]]} .. {months[target_months[-1]]}: {len(target_months)} months, {n_members} member(s)")

### Ocean totals and global-mean fluxes

The budget constraint makes the predicted ocean totals follow the surface fluxes, which are now the coupler's. So
they track ACCESS-OM2 only as well as the coupled global-mean fluxes do (second figure: coupled against ACCESS-OM2).

In [ ]:
fig, axes = plt.subplots(n_state, 1, figsize=(11, 3.4 * n_state), sharex=True, constrained_layout=True)
for v in range(n_state):   # v = 0: OHC, 1: freshwater content, 2: SST, 3: SSS
    ax = axes[v]

    # Ocean total = sum over cells of (anomaly x cell area). Land has area 0.
    member_totals = (predicted_anomaly[:, :, v] * area).sum(dim=(2, 3))   # (members, months)
    member_lines = ax.plot(dates, member_totals.T, color="tab:orange", lw=0.8, alpha=0.6)   # one line per member
    member_lines[0].set_label("emulator")
    true_total = (true_anomaly[:, v] * area).sum(dim=(1, 2))
    ax.plot(dates, true_total, color="k", lw=2, label="ACCESS-OM2")

    ax.set_title(f"Ocean-total {PROGNOSTIC[v]} anomaly")
    ax.set_ylabel(f"{units[v]} x m$^2$")
    ax.grid(alpha=0.3)
    ax.legend(frameon=False)
fig.savefig(FIG_DIR / "skill_ocean_totals.png", dpi=150, bbox_inches="tight")

In [ ]:
# Global-mean coupled fluxes against ACCESS-OM2's, yearly means (heat in W/m2, freshwater in kg/m2/s).
ocean_area_total = float(area.sum())
coupled_global_mean = coupled_totals.float() / ocean_area_total                                   # (members, months, 4)
true_global_mean = saved["forcing_total"][target_months].float() / ocean_area_total             # (months, 4)
years_of_test = dates.year.values

fig, axes = plt.subplots(1, 2, figsize=(13, 3.6), constrained_layout=True)
for ax, c in zip(axes, [heat_flux_channel, freshwater_flux_channel]):
    for member in range(coupled_global_mean.shape[0]):
        yearly = pd.Series(coupled_global_mean[member, :, c].numpy()).groupby(years_of_test).mean()
        ax.plot(yearly.index, yearly.values, color="tab:orange", lw=0.8, alpha=0.7, label="coupled" if member == 0 else None)
    yearly_true = pd.Series(true_global_mean[:, c].numpy()).groupby(years_of_test).mean()
    ax.plot(yearly_true.index, yearly_true.values, color="k", lw=2, label="ACCESS-OM2")
    ax.set_title(f"Global-mean {FORCING[c]}, yearly mean")
    ax.grid(alpha=0.3)
    ax.legend(frameon=False)
fig.savefig(FIG_DIR / "skill_global_fluxes.png", dpi=150, bbox_inches="tight")

heat_error = (coupled_global_mean[:, :, heat_flux_channel].mean(0) - true_global_mean[:, heat_flux_channel]).numpy()
print("global-mean heat flux, coupled minus ACCESS-OM2, by year (W/m2):",
      np.round(pd.Series(heat_error).groupby(years_of_test).mean().values, 2))


### Error and spread

- **RMSE of the ensemble mean**, area-weighted, against persistence and climatology. Below climatology = skill.
- **Spread**: the RMS standard deviation across members. For a reliable ensemble, spread ≈ RMSE of the
  ensemble mean (spread/error = 1); below 1 means overconfident.

In [ ]:
# The ensemble mean, (months, 4, lat, lon)
ensemble_mean = predicted_anomaly.mean(dim=0)

# Area-weighted RMSE: sqrt(sum over cells of area_weight x error^2). Each is (months, 2).
rmse = ((ensemble_mean - true_anomaly) ** 2 * area_weight).sum(dim=(2, 3)).sqrt()
rmse_persistence = ((persistence_anomaly - true_anomaly) ** 2 * area_weight).sum(dim=(2, 3)).sqrt()
rmse_climatology = (true_anomaly ** 2 * area_weight).sum(dim=(2, 3)).sqrt()   # climatology predicts zero anomaly

# Spread: the variance across members at every cell, area-averaged, square-rooted. (months, 2)
spread = (predicted_anomaly.var(dim=0) * area_weight).sum(dim=(2, 3)).sqrt()

fig, axes = plt.subplots(1, n_state, figsize=(14, 4), constrained_layout=True)
for v in range(n_state):
    ax = axes[v]
    ax.plot(dates, rmse[:, v], lw=2, label="ensemble-mean RMSE")
    ax.plot(dates, spread[:, v], lw=2, ls="--", label="ensemble spread")
    ax.plot(dates, rmse_persistence[:, v], color="0.5", lw=1.5, label="persistence")
    ax.plot(dates, rmse_climatology[:, v], color="k", lw=1.5, label="climatology")
    ax.set_title(PROGNOSTIC[v])
    ax.set_ylabel(units[v])
    ax.grid(alpha=0.3)
    ax.legend(frameon=False, fontsize=8)
fig.savefig(FIG_DIR / "skill_rmse.png", dpi=150, bbox_inches="tight")

# Scores over the first 12 months, the first 36, and the whole test. Ratio < 1 beats climatology.
scores = {}
for v, name in enumerate(PROGNOSTIC):
    scores[name] = {
        "RMSE / climatology, months 1-12": float(rmse[:12, v].mean() / rmse_climatology[:12, v].mean()),
        "RMSE / climatology, months 1-36": float(rmse[:36, v].mean() / rmse_climatology[:36, v].mean()),
        "RMSE / climatology, whole test": float(rmse[:, v].mean() / rmse_climatology[:, v].mean()),
        "spread / error, months 1-36": float(spread[:36, v].mean() / rmse[:36, v].mean()),
    }
display(pd.DataFrame(scores).T.round(3))

### Maps

Anomaly maps for one period (`"YYYY"` for a year mean, `"YYYY-MM"` for a month): ACCESS-OM2,
the ensemble mean, one member, and the error of the ensemble mean.
The ensemble mean is smooth (it averages out what the members disagree on); a single member should look
as detailed as the truth.

In [ ]:
PERIOD = "2010"   # a year ("2010") or a month ("2010-12")

# Positions of the months in PERIOD: dates whose "YYYY-MM" starts with PERIOD.
in_period = [i for i, date in enumerate(dates) if date.strftime("%Y-%m").startswith(PERIOD)]

titles = ["ACCESS-OM2", "ensemble mean", "member 1", "ensemble mean - ACCESS-OM2"]
fig, axes = plt.subplots(n_state, 4, figsize=(18.4, 3.2 * n_state), squeeze=False, constrained_layout=True)
for v in range(n_state):
    # Average over the months of PERIOD. Land set to NaN so it shows blank.
    truth_map = true_anomaly[in_period, v].mean(dim=0).numpy()
    mean_map = predicted_anomaly[:, in_period, v].mean(dim=(0, 1)).numpy()
    member_map = predicted_anomaly[0, in_period, v].mean(dim=0).numpy()
    error_map = mean_map - truth_map
    maps = [np.where(ocean, field, np.nan) for field in [truth_map, mean_map, member_map, error_map]]

    # One colour scale per row: 99th percentile of |truth|.
    limit = np.nanquantile(np.abs(maps[0]), 0.99)
    for column in range(4):
        image = axes[v, column].pcolormesh(longitude, latitude, maps[column], cmap="RdBu_r",
                                           vmin=-limit, vmax=limit, shading="auto")
        axes[v, column].set_title(f"{titles[column]} | {PERIOD}", fontsize=10)
    fig.colorbar(image, ax=axes[v], shrink=0.85, label=f"{PROGNOSTIC[v]} anomaly ({units[v]})")
fig.savefig(FIG_DIR / f"skill_maps_{PERIOD}.png", dpi=150, bbox_inches="tight")

### Zonal spectra

Energy per zonal wavenumber in the first and last year of the test, ACCESS-OM2 against
one member (the quantity the spectral loss matches).
Emulator below the truth at long wavelengths = anomalies fading; above it at short wavelengths =
grid-scale noise growing.

The first cell below is the copied spectrum code; the second plots it.

In [ ]:
# ==================================================================================================
# IDENTICAL CODE EXISTS IN src/Experiment/evaluation.py
#   zonal_spectrum
# Copied verbatim so the notebook shows everything it runs. Editing it here does NOT change src/
# (which the test suite checks), and vice versa.
# ==================================================================================================

def zonal_spectrum(anomaly, row_length_km):
    """
    Time-mean along-row power spectral density of an anomaly (time, latitude, longitude) array, land
    NaN or 0, against physical wavenumber, on the same bands as the spectral loss
    (zonal_wavenumber_bands), over every grid row (row_length_km: (latitude,), grid_row_length_km).
    Returns (k in cycles/km, PSD in units^2 km). PSD is the area-weighted mean along-row variance per
    unit k, so it reads the same at 1 deg and 1/10 deg.
    """
    band, power_weight, k = zonal_wavenumber_bands(row_length_km, anomaly.shape[-1])
    x = np.nan_to_num(np.asarray(anomaly, dtype=float))
    # Power of every (row, harmonic), time-mean, without the row mean; then summed into the bands.
    power = (np.abs(np.fft.rfft(x, axis=-1)) ** 2).mean(0)[:, 1:] * power_weight
    length_share = np.asarray(row_length_km, dtype=float) / (2 * np.pi * EARTH_RADIUS_KM)
    energy = np.bincount(band, power.ravel(), minlength=k.size) / length_share.sum()
    # Energy per band / band width in k = density.
    half = 0.5 / SPECTRAL_BANDS_PER_DECADE
    return k, energy / (k * (10.0**half - 10.0**-half))


In [ ]:
fig, axes = plt.subplots(n_state, 2, figsize=(12, 3.6 * n_state), squeeze=False, constrained_layout=True)
for v in range(n_state):
    for column, (label, year) in enumerate([("first year", slice(0, 12)), ("last year", slice(-12, None))]):
        ax = axes[v, column]

        # zonal_spectrum takes a (months, lat, lon) array; multiplying by ocean (True/False) sets land to 0.
        wavenumber, truth_spectrum = zonal_spectrum(true_anomaly[year, v].numpy() * ocean, row_length_km)
        wavenumber, emulator_spectrum = zonal_spectrum(predicted_anomaly[0, year, v].numpy() * ocean, row_length_km)   # member 1

        ax.loglog(wavenumber, truth_spectrum, "-", lw=2, label="ACCESS-OM2")
        ax.loglog(wavenumber, emulator_spectrum, "--", lw=2, label="member 1")
        ax.set_title(f"{PROGNOSTIC[v]}: {label}")
        ax.set_xlabel("zonal wavenumber (cycles/km)")
        ax.set_ylabel("power spectral density")
        ax.grid(alpha=0.3, which="both")
        ax.legend(frameon=False)
fig.savefig(FIG_DIR / "skill_spectra.png", dpi=150, bbox_inches="tight")

## 11. Stability: 20-year coupled control

Repeat the May 1990 – April 1991 atmosphere and sea ice 20 times, **coupled** (fluxes from the coupler and
the emulator's own surface), starting from the true state of May 1990 (one member).
This is the repeat year of ACCESS-OM2's RYF runs (JRA55-do RYF 1990-91), chosen there for its near-neutral
climate. Coupled, the net surface flux is whatever the coupler computes from the emulator's surface, so the ocean
totals can drift: watch the bottom panels and the global-mean heat flux.
There is no truth to compare with; the question is whether the state stays in the range ACCESS-OM2 visits.

- **Largest |z| over the ocean** each month, on a log scale. ACCESS-OM2's own largest |z| over 2005–2018 is
  the dashed line. Exponential growth shows as a straight rising line.
- **Ocean totals**: they follow the coupled fluxes, so a slow drift of the coupled system shows here.
- **Global-mean coupled heat flux**, yearly: ~0 W/m2 for a balanced ocean.

The coupled prototypes (statistical coupler) reached 38 (seed 42) and 212 (seed 43) by year 50. With the physical
coupler and no salinity restoring (runs `unet_deterministic_coupled_seed42`, `_seed43`, 50 years), the largest |z| passed
20 in year 14 (seed 42) and year 5 (seed 43) and reached ~100 by year 50; only SST (seed 43) stayed in range.

In [ ]:
first = month_index[CONTROL_YEAR[0]]   # May 1990: the starting state
last = month_index[CONTROL_YEAR[1]]    # April 1991

# Predicting month t+1 uses the forcing of month t+1, so one year of steps uses the forcing of
# June 1990 .. May 1991. Repeat that year CONTROL_YEARS times.
one_year = list(range(first + 1, last + 2))
control_months = torch.tensor(one_year * CONTROL_YEARS)   # forcing month (time index) of every step

# The coupler for the control year: free the test months' weather, read June 1990 .. May 1991.
del coupler, test_inputs
torch.cuda.empty_cache()
control_inputs = read_coupler_inputs([months[i] for i in one_year], one_year, mask, DEVICE, WEATHER_DEVICE)
coupler = PhysicalCoupler(
    control_inputs, true_state_z=state_z, true_fluxes=true_fluxes, surface_channels=surface_channels,
    state_mean=state_mean, state_std=state_std, forcing_mean=forcing_mean, forcing_scale=flux_scale, mask=mask, area=area,
)

z_control, control_totals = coupled_rollout(
    ensemble_model,
    coupler,
    prior=state_z[first - 1 : first + 1],               # true April and May 1990
    target_indices=control_months,                      # the 1990-91 atmosphere and sea ice, over and over
    initial_index=first,
    column_budget=column_budget,
    constraint=constraint,
    initial_forcing=forcing_z[first],                     # ACCESS-OM2's forcing input of the starting month
    initial_flux_totals=saved["forcing_total"][first],  # ACCESS-OM2's ocean-total fluxes of May 1990
    members=1,
)
z_control = z_control[0]                # the one member: (steps, 4, lat, lon) z-scores
control_totals = control_totals[0]      # (steps, 4) ocean-total coupled fluxes

# Largest |z| over the ocean at every step, (steps, 2). Land is set to 0 so it never counts.
land = mask == 0
largest_z = z_control.abs().masked_fill(land, 0).amax(dim=(2, 3))
# ACCESS-OM2's own largest |z| over the skill-test period, (2,), as a reference.
largest_z_truth = state_z[target_months].abs().masked_fill(land, 0).amax(dim=(0, 2, 3))

# Ocean-total anomaly at every step: z x std x area, summed over cells. (steps, 2)
control_total = (z_control * state_std[control_months] * area).sum(dim=(2, 3))

years = np.arange(1, len(control_months) + 1) / 12   # time axis in years

fig, axes = plt.subplots(2, n_state, figsize=(14, 7), sharex=True, constrained_layout=True)
for v in range(n_state):
    axes[0, v].semilogy(years, largest_z[:, v], lw=1.5)
    axes[0, v].axhline(largest_z_truth[v], color="k", ls="--", lw=1, label="ACCESS-OM2 largest, 2005-2018")
    axes[0, v].set_title(f"{PROGNOSTIC[v]}: largest |z| over the ocean")
    axes[0, v].legend(frameon=False, fontsize=8)

    axes[1, v].plot(years, control_total[:, v], lw=1.5)
    axes[1, v].set_title(f"{PROGNOSTIC[v]}: ocean-total anomaly ({units[v]} x m$^2$)")
    axes[1, v].set_xlabel("year of control run")

    axes[0, v].grid(alpha=0.3, which="both")
    axes[1, v].grid(alpha=0.3)
fig.savefig(FIG_DIR / f"control_{CONTROL_YEARS}yr.png", dpi=150, bbox_inches="tight")

# Global-mean coupled heat flux, one value per year of the control (W/m2): what the ocean-total heat drifts by.
yearly_heat_flux = control_totals[:, heat_flux_channel].reshape(CONTROL_YEARS, 12).mean(dim=1) / area.sum()
fig, ax = plt.subplots(figsize=(7, 3), constrained_layout=True)
ax.plot(np.arange(1, CONTROL_YEARS + 1), yearly_heat_flux, lw=1.5)
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("year of control run")
ax.set_ylabel("W/m$^2$")
ax.set_title("Global-mean coupled heat flux into the ocean, yearly mean")
ax.grid(alpha=0.3)
fig.savefig(FIG_DIR / f"control_{CONTROL_YEARS}yr_heat_flux.png", dpi=150, bbox_inches="tight")
print("global-mean coupled heat flux, mean of each decade (W/m2):", np.round(yearly_heat_flux.reshape(-1, 10).mean(dim=1).numpy(), 2))

# When (if ever) did any cell first pass |z| = 20? The years of every step where one did:
years_above_20 = years[(largest_z > 20).any(dim=1).numpy()]
if len(years_above_20) > 0:
    print(f"largest |z| first above 20 in year {years_above_20[0]:.1f}")
else:
    print("largest |z| never above 20")

## Notes

- Started from the settings of the `crps_ensemble` emulator of `runs/intercomparison_90ep_v3` (seed 42);
  since then: SST, SSS and the surface current predicted, the forcing divided by one scale per flux, and both
  stages and both tests coupled to ACCESS-OM2's own bulk formulae (section 5; it replaced a statistical coupler, FluxNet,
  whose evaporation was biased by +2.7e-7 kg/m2/s over 2005-2018).
- The surface current comes from data v5 (`/g/data/nm47/txs156/OM2-emulator/data/v5_scripts/build_v5.py`).
- The deterministic version is `Ocean_Emulator_UNet.ipynb`; stage 1 is identical in both.
- Figures are saved to `figures/unet_crps_coupled/`.
- The copied cells (`IDENTICAL CODE EXISTS IN src/...`) do not follow later changes to `src/`.